# Uber NYC Ride-Hailing: Geographic Evolution 2018-2025

Comparative spatial analysis of Uber pickup and dropoff patterns across New York City taxi zones, using K-means clustering on geographic coordinates, Lorenz curve concentration measures, and Local Indicators of Spatial Association (LISA) to quantify how ride-hailing demand has redistributed over seven years.

---
## 1. Setup and Configuration

In [12]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.stats import ks_2samp, levene
from math import radians, cos, sin, asin, sqrt
import geopandas as gpd
import libpysal
from esda.moran import Moran, Moran_Local
import json
import gc
import os

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR = '/Users/leoss/Desktop/Portfolio/Website-/'
DATA_DIR = BASE_DIR + 'projects/uber/data/'
OUTPUT_DIR = BASE_DIR + 'projects/uber/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_2018 = DATA_DIR + 'fhv_tripdata_2018-01.parquet'
PATH_2025 = DATA_DIR + 'fhvhv_tripdata_2025-01.parquet'
PATH_CENTROIDS = DATA_DIR + 'zone_centroids.csv'
PATH_SHAPEFILE = DATA_DIR + 'taxi_zones/taxi_zones.shp'

# ── Analysis parameters ────────────────────────────────────────────────────
SAMPLE_SIZE = 20_000_000
N_CLUSTERS = 6

UBER_2018_BASES = ['B02512', 'B02598', 'B02617', 'B02682', 'B02764', 'B02765', 'B02835', 'B02836']
UBER_2025_LICENSE = 'HV0003'

AIRPORT_ZONE_IDS = {132, 138, 1}
AIRPORT_LABELS = {132: 'JFK', 138: 'LaGuardia', 1: 'Newark (EWR)'}

# ── Unified style system ──────────────────────────────────────────────────
STYLE = {
    'font_family': 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size': 11,
    'axis_title_size': 13,
    'legend_size': 11,
    'template': 'plotly_white',
    'plot_bg': 'rgba(0,0,0,0)',
    'paper_bg': 'white',
    'chart_height': 550,
    'margin_default': dict(l=60, r=40, t=20, b=50),
    'margin_map': dict(l=0, r=0, t=20, b=0),
    'grid_color': '#e5e7eb',
    'grid_width': 0.5,
    'hover_font_size': 13,
    'hover_font_color': '#1a2744',
    # Year comparison palette
    'year_2018': '#ff6b6b',
    'year_2025': '#4ecdc4',
    # Cluster palette (6 clusters)
    'cluster_colors': ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4'],
    # Borough palette
    'borough_colors': {
        'Manhattan': '#4363d8', 'Brooklyn': '#3cb44b', 'Queens': '#f58231',
        'Bronx': '#e6194b', 'Staten Island': '#911eb4', 'EWR': '#42d4f4',
    },
    # LISA palette
    'lisa_colors': {
        'HH': '#d7191c', 'LL': '#2c7bb6', 'HL': '#fdae61',
        'LH': '#abd9e9', 'ns': '#e8e8e8',
    },
    'lisa_labels': {
        'HH': 'High-High (hot spot)', 'LL': 'Low-Low (cold spot)',
        'HL': 'High-Low (outlier)', 'LH': 'Low-High (outlier)',
        'ns': 'Not significant',
    },
    # Map defaults
    'map_style': 'carto-positron-nolabels',
    'map_center': {'lat': 40.7128, 'lon': -73.9352},
    'map_zoom': 9.5,
}


def base_layout(height=None, width=None, **kwargs):
    """Standard layout applied to every chart."""
    layout = dict(
        title='',
        font=dict(family=STYLE['font_family']),
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        height=height or STYLE['chart_height'],
        margin=STYLE['margin_default'],
        hoverlabel=dict(
            font_size=STYLE['hover_font_size'],
            font_color=STYLE['hover_font_color'],
        ),
    )
    if width:
        layout['width'] = width
    layout.update(kwargs)
    return layout


def styled_axis(**kwargs):
    """Standard axis styling."""
    return dict(
        tickfont=dict(size=STYLE['tick_size']),
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'],
        gridwidth=STYLE['grid_width'],
        **kwargs,
    )


def save_html(fig, filename):
    """Save figure as CDN-loaded HTML with mode bar suppressed."""
    fig.write_html(
        OUTPUT_DIR + filename,
        include_plotlyjs='cdn',
        config={'displayModeBar': False},
    )
    print(f"  Saved: {filename}")


print(f"Configuration:")
print(f"  Sample size: {SAMPLE_SIZE:,} trips per year")
print(f"  Clusters: {N_CLUSTERS}")
print(f"  Output: {OUTPUT_DIR}")

Configuration:
  Sample size: 20,000,000 trips per year
  Clusters: 6
  Output: /Users/leoss/Desktop/Portfolio/Website-/projects/uber/outputs/


---
## 2. Helper Functions

In [13]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two points in km."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * 6371 * asin(sqrt(a))


def name_cluster_by_location(center_lat, center_lon, zone_centroids):
    """Name cluster based on nearest major zone."""
    zone_distances = np.sqrt(
        (zone_centroids['latitude'] - center_lat) ** 2 +
        (zone_centroids['longitude'] - center_lon) ** 2
    )
    nearest_idx = zone_distances.idxmin()
    nearest_zone = zone_centroids.iloc[nearest_idx]
    return f"{nearest_zone['borough']}: {nearest_zone['zone_name']}"


def match_clusters_by_proximity(centers_2018, centers_2025):
    """Match 2018 to 2025 clusters by geographic proximity (greedy nearest)."""
    distances = cdist(centers_2018, centers_2025, metric='cityblock')
    matches = {}
    used_2025 = set()
    pairs = []
    for i in range(len(centers_2018)):
        for j in range(len(centers_2025)):
            pairs.append((i, j, distances[i, j]))
    pairs.sort(key=lambda x: x[2])
    for i, j, dist in pairs:
        if i not in matches and j not in used_2025:
            matches[i] = j
            used_2025.add(j)
    return matches


def lorenz_data(df, zone_col='PU_zone_id'):
    """Return (cumulative share of zones, cumulative share of trips)."""
    zone_counts = df.groupby(zone_col).size().sort_values().values
    cum_zones = np.arange(1, len(zone_counts) + 1) / len(zone_counts)
    cum_trips = np.cumsum(zone_counts) / zone_counts.sum()
    return cum_zones, cum_trips


def gini_from_lorenz(cum_zones, cum_trips):
    """Gini coefficient via trapezoidal integration under the Lorenz curve."""
    area_under = np.trapz(cum_trips, cum_zones)
    return 1 - 2 * area_under


def merge_zone_info(df, zone_col, centroids, prefix):
    """Merge zone centroid info onto a dataframe, with column prefix."""
    merged = df.merge(
        centroids[['zone_id', 'zone_name', 'borough', 'latitude', 'longitude']],
        left_on=zone_col, right_on='zone_id', how='left'
    )
    merged = merged.rename(columns={
        'zone_id': f'{prefix}_zone_id',
        'zone_name': f'{prefix}_zone_name',
        'borough': f'{prefix}_borough',
        'latitude': f'{prefix}_lat',
        'longitude': f'{prefix}_lon',
    })
    return merged


def mismatch_ratios(df, pu_col='PU_zone_id', do_col='DO_zone_id'):
    """Compute PU/(PU+DO) ratio per zone. 0.5 = perfectly balanced."""
    pu = df.groupby(pu_col).size().rename('pu')
    do = df.dropna(subset=[do_col]).groupby(do_col).size().rename('do')
    combined = pd.concat([pu, do], axis=1).fillna(0)
    combined['ratio'] = combined['pu'] / (combined['pu'] + combined['do'])
    combined['total'] = combined['pu'] + combined['do']
    return combined


def short_cluster_label(full_name):
    """Extract borough from 'Borough: Zone Name' format."""
    return full_name.split(':')[0].strip()


def make_short_labels(cluster_names, n_clusters):
    """Create short labels, appending (2), (3) for duplicate boroughs."""
    labels = {}
    seen = {}
    for i in range(n_clusters):
        base = short_cluster_label(cluster_names[i])
        if base in seen:
            seen[base] += 1
            labels[i] = f"{base} ({seen[base]})"
        else:
            seen[base] = 1
            labels[i] = base
    return labels

---
## 3. Load Zone Centroids and Shapefile

In [14]:
zone_centroids = pd.read_csv(PATH_CENTROIDS)
print(f"Loaded {len(zone_centroids)} zone centroids")

gdf_raw = gpd.read_file(PATH_SHAPEFILE)
gdf_raw = gdf_raw.to_crs(epsg=4326)
taxi_zones_geo_4326 = json.loads(gdf_raw.to_json())

for f in taxi_zones_geo_4326['features']:
    f['properties']['LocationID'] = str(int(f['properties']['LocationID']))

all_zone_ids = [f['properties']['LocationID'] for f in taxi_zones_geo_4326['features']]
print(f"Loaded {len(all_zone_ids)} taxi zone geometries")

Loaded 263 zone centroids
Loaded 263 taxi zone geometries


---
## 4. 2018 Uber Data: Load, Process, Cluster

In [15]:
print("[4.1] Loading 2018 data...")
table_2018 = pq.read_table(PATH_2018, columns=[])
total_2018 = table_2018.num_rows
print(f"  Total rows: {total_2018:,}")

columns_2018 = ['pickup_datetime', 'PUlocationID', 'DOlocationID', 'dispatching_base_num']
table_2018 = pq.read_table(PATH_2018, columns=columns_2018)

df_2018_full = table_2018.to_pandas()
df_2018_full = df_2018_full[df_2018_full['dispatching_base_num'].isin(UBER_2018_BASES)].copy()

uber_count_2018 = len(df_2018_full)
print(f"  Uber trips: {uber_count_2018:,} ({100 * uber_count_2018 / total_2018:.1f}%)")

df_2018 = df_2018_full.sample(n=min(SAMPLE_SIZE, uber_count_2018), random_state=42)
del df_2018_full, table_2018
gc.collect()

print("[4.2] Processing temporal features...")
df_2018['pickup_datetime'] = pd.to_datetime(df_2018['pickup_datetime'])
df_2018['hour'] = df_2018['pickup_datetime'].dt.hour
df_2018['day_of_week'] = df_2018['pickup_datetime'].dt.dayofweek
df_2018['day_name'] = df_2018['pickup_datetime'].dt.day_name()

print("[4.3] Merging zone info...")
df_2018 = df_2018.dropna(subset=['PUlocationID'])
df_2018['PUlocationID'] = df_2018['PUlocationID'].astype(int)
df_2018 = merge_zone_info(df_2018, 'PUlocationID', zone_centroids, 'PU')
df_2018 = df_2018.dropna(subset=['PU_lat', 'PU_lon'])

df_2018['DOlocationID'] = pd.to_numeric(df_2018['DOlocationID'], errors='coerce')
df_2018.loc[df_2018['DOlocationID'].notna(), 'DOlocationID'] = \
    df_2018.loc[df_2018['DOlocationID'].notna(), 'DOlocationID'].astype(int)
df_2018 = merge_zone_info(df_2018, 'DOlocationID', zone_centroids, 'DO')

print("[4.4] Clustering on pickup coordinates...")
coords_2018 = df_2018[['PU_lat', 'PU_lon']].values
kmeans_2018 = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_2018['PU_cluster'] = kmeans_2018.fit_predict(coords_2018)

cluster_names_2018 = {}
for i in range(N_CLUSTERS):
    lat, lon = kmeans_2018.cluster_centers_[i]
    cluster_names_2018[i] = name_cluster_by_location(lat, lon, zone_centroids)

df_2018['PU_cluster_name'] = df_2018['PU_cluster'].map(cluster_names_2018)

do_mask_2018 = df_2018['DO_lat'].notna() & df_2018['DO_lon'].notna()
df_2018.loc[do_mask_2018, 'DO_cluster'] = kmeans_2018.predict(
    df_2018.loc[do_mask_2018, ['DO_lat', 'DO_lon']].values
)
df_2018['DO_cluster_name'] = df_2018['DO_cluster'].map(cluster_names_2018)

print("  Cluster distribution (2018):")
for i in range(N_CLUSTERS):
    count = (df_2018['PU_cluster'] == i).sum()
    pct = 100 * count / len(df_2018)
    print(f"    {cluster_names_2018[i]}: {count:>7,} ({pct:>5.1f}%)")

[4.1] Loading 2018 data...
  Total rows: 19,808,094
  Uber trips: 4,502,999 (22.7%)
[4.2] Processing temporal features...
[4.3] Merging zone info...
[4.4] Clustering on pickup coordinates...
  Cluster distribution (2018):
    Bronx: East Tremont: 477,684 ( 10.6%)
    Manhattan: Gramercy: 1,883,665 ( 41.7%)
    Brooklyn: Ocean Hill: 626,926 ( 13.9%)
    Queens: Briarwood/Jamaica Hills: 373,460 (  8.3%)
    Brooklyn: Borough Park: 388,956 (  8.6%)
    Manhattan: Yorkville East: 769,118 ( 17.0%)


---
## 5. 2025 Uber Data: Load, Process, Cluster

In [16]:
print("[5.1] Loading 2025 data...")
table_2025 = pq.read_table(PATH_2025, columns=[])
total_2025 = table_2025.num_rows
print(f"  Total rows: {total_2025:,}")

columns_2025 = ['pickup_datetime', 'PULocationID', 'DOLocationID', 'hvfhs_license_num']
table_2025 = pq.read_table(PATH_2025, columns=columns_2025)

df_2025_full = table_2025.to_pandas()
df_2025_full = df_2025_full[df_2025_full['hvfhs_license_num'] == UBER_2025_LICENSE].copy()

uber_count_2025 = len(df_2025_full)
print(f"  Uber trips: {uber_count_2025:,} ({100 * uber_count_2025 / total_2025:.1f}%)")

df_2025 = df_2025_full.sample(n=min(SAMPLE_SIZE, uber_count_2025), random_state=42)
del df_2025_full, table_2025
gc.collect()

print("[5.2] Processing temporal features...")
df_2025['pickup_datetime'] = pd.to_datetime(df_2025['pickup_datetime'])
df_2025['hour'] = df_2025['pickup_datetime'].dt.hour
df_2025['day_of_week'] = df_2025['pickup_datetime'].dt.dayofweek
df_2025['day_name'] = df_2025['pickup_datetime'].dt.day_name()

print("[5.3] Merging zone info...")
df_2025 = df_2025.dropna(subset=['PULocationID'])
df_2025['PULocationID'] = df_2025['PULocationID'].astype(int)
df_2025 = merge_zone_info(df_2025, 'PULocationID', zone_centroids, 'PU')
df_2025 = df_2025.dropna(subset=['PU_lat', 'PU_lon'])

df_2025['DOLocationID'] = pd.to_numeric(df_2025['DOLocationID'], errors='coerce')
df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'] = \
    df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'].astype(int)
df_2025 = merge_zone_info(df_2025, 'DOLocationID', zone_centroids, 'DO')

print("[5.4] Clustering on pickup coordinates...")
coords_2025 = df_2025[['PU_lat', 'PU_lon']].values
kmeans_2025 = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_2025['PU_cluster'] = kmeans_2025.fit_predict(coords_2025)

cluster_names_2025 = {}
for i in range(N_CLUSTERS):
    lat, lon = kmeans_2025.cluster_centers_[i]
    cluster_names_2025[i] = name_cluster_by_location(lat, lon, zone_centroids)

df_2025['PU_cluster_name'] = df_2025['PU_cluster'].map(cluster_names_2025)

do_mask_2025 = df_2025['DO_lat'].notna() & df_2025['DO_lon'].notna()
df_2025.loc[do_mask_2025, 'DO_cluster'] = kmeans_2025.predict(
    df_2025.loc[do_mask_2025, ['DO_lat', 'DO_lon']].values
)
df_2025['DO_cluster_name'] = df_2025['DO_cluster'].map(cluster_names_2025)

print("  Cluster distribution (2025):")
for i in range(N_CLUSTERS):
    count = (df_2025['PU_cluster'] == i).sum()
    pct = 100 * count / len(df_2025)
    print(f"    {cluster_names_2025[i]}: {count:>7,} ({pct:>5.1f}%)")

[5.1] Loading 2025 data...
  Total rows: 20,405,666
  Uber trips: 15,356,455 (75.3%)
[5.2] Processing temporal features...
[5.3] Merging zone info...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/4062608121.py:32: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[170 265  79 ... 213 248 263]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.



[5.4] Clustering on pickup coordinates...
  Cluster distribution (2025):
    Bronx: Claremont/Bathgate: 2,714,803 ( 17.6%)
    Queens: Baisley Park: 1,358,465 (  8.8%)
    Staten Island: Westerleigh: 241,368 (  1.6%)
    Manhattan: Murray Hill: 5,992,333 ( 38.8%)
    Brooklyn: Prospect-Lefferts Gardens: 3,282,505 ( 21.2%)
    Queens: Elmhurst: 1,857,722 ( 12.0%)


---
## 6. Cross-Year Cluster Matching

Clusters are fitted independently for each year. To enable meaningful comparison, 2018 and 2025 clusters are matched by greedy nearest-centroid proximity (cityblock distance on lat/lon), and a shared color mapping is imposed so that geographically corresponding clusters share the same color across all charts.

In [17]:
centers_2018 = kmeans_2018.cluster_centers_
centers_2025 = kmeans_2025.cluster_centers_

cluster_matches = match_clusters_by_proximity(centers_2018, centers_2025)

print("Cluster matches (2018 -> 2025):")
for idx_2018, idx_2025 in sorted(cluster_matches.items()):
    lat1, lon1 = centers_2018[idx_2018]
    lat2, lon2 = centers_2025[idx_2025]
    dist = haversine_km(lat1, lon1, lat2, lon2)
    print(f"  {cluster_names_2018[idx_2018]} -> {cluster_names_2025[idx_2025]} ({dist:.2f} km)")

# Build color maps: matched clusters share the same color
cluster_color_map_2018 = {
    cluster_names_2018[i]: STYLE['cluster_colors'][i % len(STYLE['cluster_colors'])]
    for i in range(N_CLUSTERS)
}
cluster_color_map_2025 = {
    cluster_names_2025[idx_25]: STYLE['cluster_colors'][idx_18 % len(STYLE['cluster_colors'])]
    for idx_18, idx_25 in cluster_matches.items()
}

short_labels_2018 = make_short_labels(cluster_names_2018, N_CLUSTERS)
short_labels_2025 = make_short_labels(cluster_names_2025, N_CLUSTERS)

Cluster matches (2018 -> 2025):
  Bronx: East Tremont -> Bronx: Claremont/Bathgate (0.72 km)
  Manhattan: Gramercy -> Manhattan: Murray Hill (1.22 km)
  Brooklyn: Ocean Hill -> Brooklyn: Prospect-Lefferts Gardens (3.03 km)
  Queens: Briarwood/Jamaica Hills -> Queens: Baisley Park (2.59 km)
  Brooklyn: Borough Park -> Staten Island: Westerleigh (11.62 km)
  Manhattan: Yorkville East -> Queens: Elmhurst (6.14 km)


---
## 7. Cluster Maps

Choropleth maps showing the dominant pickup cluster per taxi zone for each year, followed by centroid shift vectors illustrating how cluster centers migrated between 2018 and 2025.

In [18]:
# ── 2018 Cluster Map ──────────────────────────────────────────────────────
print("[7.1] 2018 cluster map...")

zone_clusters = df_2018.groupby('PU_zone_id').agg({
    'PU_cluster': lambda x: x.mode()[0],
    'PU_cluster_name': lambda x: x.mode()[0],
    'PU_zone_name': 'first',
    'PU_borough': 'first'
}).reset_index()
zone_clusters.columns = ['zone_id', 'cluster', 'cluster_name', 'zone_name', 'borough']
zone_clusters['zone_id'] = zone_clusters['zone_id'].astype(float).astype(int).astype(str)

zone_ids_with_data = set(zone_clusters['zone_id'].values)
filtered_geojson = {
    'type': 'FeatureCollection',
    'features': [f for f in taxi_zones_geo_4326['features']
                 if f['properties']['LocationID'] in zone_ids_with_data]
}

fig = go.Figure()

# Background: all zones in light grey
fig.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[1] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))

# Overlay: cluster-colored zones
fig_tmp = px.choropleth_map(
    zone_clusters, geojson=filtered_geojson,
    locations='zone_id', featureidkey='properties.LocationID',
    color='cluster_name', color_discrete_map=cluster_color_map_2018,
    map_style=STYLE['map_style'], zoom=STYLE['map_zoom'],
    center=STYLE['map_center'], opacity=0.95,
    hover_data={'zone_id': False, 'zone_name': True, 'borough': True, 'cluster_name': True},
    labels={'cluster_name': 'Cluster'}
)
fig_tmp.update_traces(marker=dict(line=dict(width=0.8, color='rgba(255,255,255,0.8)')))
for trace in fig_tmp.data:
    fig.add_trace(trace)

# Centroid markers
for i, (lat, lon) in enumerate(centers_2018):
    name = cluster_names_2018[i]
    fig.add_trace(go.Scattermap(
        lat=[lat], lon=[lon], mode='markers+text',
        marker=dict(size=0, color='black', opacity=0.85),
        text=str(i), textfont=dict(size=10, color='white', family='Arial Black'),
        textposition='middle center', name=name, showlegend=False,
        hovertemplate=f'<b>Cluster {i}</b><br>{name}<extra></extra>',
    ))

fig.update_layout(
    **base_layout(height=650, width=1100, margin=STYLE['margin_map']),
    legend=dict(
        title="Clusters", yanchor="top", y=0.99, xanchor="left", x=0.01,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#dde1e7", borderwidth=1,
        font=dict(size=STYLE['legend_size'], family=STYLE['font_family']),
    ),
    map=dict(style=STYLE['map_style'], center=STYLE['map_center'], zoom=STYLE['map_zoom']),
)

save_html(fig, 'cluster_map_2018.html')

[7.1] 2018 cluster map...
  Saved: cluster_map_2018.html


In [19]:
# ── 2025 Cluster Map ──────────────────────────────────────────────────────
print("[7.2] 2025 cluster map...")

zone_clusters_2025 = df_2025.groupby('PU_zone_id').agg({
    'PU_cluster': lambda x: x.mode()[0],
    'PU_cluster_name': lambda x: x.mode()[0],
    'PU_zone_name': 'first',
    'PU_borough': 'first'
}).reset_index()
zone_clusters_2025.columns = ['zone_id', 'cluster', 'cluster_name', 'zone_name', 'borough']
zone_clusters_2025['zone_id'] = zone_clusters_2025['zone_id'].astype(float).astype(int).astype(str)

zone_ids_2025 = set(zone_clusters_2025['zone_id'].values)
filtered_geojson_2025 = {
    'type': 'FeatureCollection',
    'features': [f for f in taxi_zones_geo_4326['features']
                 if f['properties']['LocationID'] in zone_ids_2025]
}

fig_2025 = go.Figure()

fig_2025.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[1] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))

fig_tmp_2025 = px.choropleth_map(
    zone_clusters_2025, geojson=filtered_geojson_2025,
    locations='zone_id', featureidkey='properties.LocationID',
    color='cluster_name', color_discrete_map=cluster_color_map_2025,
    map_style=STYLE['map_style'], zoom=STYLE['map_zoom'],
    center=STYLE['map_center'], opacity=0.95,
    hover_data={'zone_id': False, 'zone_name': True, 'borough': True, 'cluster_name': True},
    labels={'cluster_name': 'Cluster'}
)
fig_tmp_2025.update_traces(marker=dict(line=dict(width=0.8, color='rgba(255,255,255,0.8)')))
for trace in fig_tmp_2025.data:
    fig_2025.add_trace(trace)

for i, (lat, lon) in enumerate(centers_2025):
    name = cluster_names_2025[i]
    fig_2025.add_trace(go.Scattermap(
        lat=[lat], lon=[lon], mode='markers+text',
        marker=dict(size=0, color='black', opacity=0.85),
        text=str(i), textfont=dict(size=11, color='white', family='Arial Black'),
        textposition='middle center', name=name, showlegend=False,
        hovertemplate=f'<b>Cluster {i}</b><br>{name}<extra></extra>',
    ))

fig_2025.update_layout(
    **base_layout(height=650, width=1100, margin=STYLE['margin_map']),
    legend=dict(
        title="Clusters", yanchor="top", y=0.99, xanchor="left", x=0.01,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#dde1e7", borderwidth=1,
        font=dict(size=STYLE['legend_size'], family=STYLE['font_family']),
    ),
    map=dict(style=STYLE['map_style'], center=STYLE['map_center'], zoom=STYLE['map_zoom']),
)

save_html(fig_2025, 'cluster_map_2025.html')

[7.2] 2025 cluster map...


  Saved: cluster_map_2025.html


In [20]:
# ── Cluster Centroid Shifts ───────────────────────────────────────────────
print("[7.3] Cluster shifts map...")

shift_data = []
for idx_18, idx_25 in cluster_matches.items():
    lat1, lon1 = centers_2018[idx_18]
    lat2, lon2 = centers_2025[idx_25]
    shift_data.append({
        'idx_18': idx_18, 'idx_25': idx_25,
        'name_18': cluster_names_2018[idx_18],
        'name_25': cluster_names_2025[idx_25],
        'dist_km': haversine_km(lat1, lon1, lat2, lon2)
    })

max_shift = max(s['dist_km'] for s in shift_data)
min_shift = min(s['dist_km'] for s in shift_data)

fig_shift = go.Figure()

fig_shift.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[1] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.35, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))

# Shift vectors
for sa in shift_data:
    lat1, lon1 = centers_2018[sa['idx_18']]
    lat2, lon2 = centers_2025[sa['idx_25']]
    if max_shift > min_shift:
        width = 1.5 + 3.5 * (sa['dist_km'] - min_shift) / (max_shift - min_shift)
    else:
        width = 3
    fig_shift.add_trace(go.Scattermap(
        lat=[lat1, lat2], lon=[lon1, lon2],
        mode='lines', line=dict(width=width, color='#333'),
        showlegend=False,
        hovertemplate=(
            f"<b>{sa['name_18']}</b> -> <b>{sa['name_25']}</b>"
            f"<br>Shift: {sa['dist_km']:.2f} km<extra></extra>"
        )
    ))

# 2018 centroids (circles)
for i in range(len(centers_2018)):
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2018[i, 0]], lon=[centers_2018[i, 1]],
        mode='markers',
        marker=dict(size=16, color=STYLE['cluster_colors'][i % len(STYLE['cluster_colors'])], opacity=0.9),
        name=f'2018: {cluster_names_2018[i]}',
        legendgroup='2018', legendgrouptitle_text='2018 Centroids',
        hovertemplate=f'<b>2018 Cluster {i}</b><br>{cluster_names_2018[i]}<extra></extra>',
    ))

# 2025 centroids (squares)
for i in range(len(centers_2025)):
    matched_18 = [k for k, v in cluster_matches.items() if v == i]
    color_idx = matched_18[0] if matched_18 else i
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2025[i, 0]], lon=[centers_2025[i, 1]],
        mode='markers',
        marker=dict(
            size=16,
            color=STYLE['cluster_colors'][color_idx % len(STYLE['cluster_colors'])],
            opacity=0.9, symbol='square',
        ),
        name=f'2025: {cluster_names_2025[i]}',
        legendgroup='2025', legendgrouptitle_text='2025 Centroids',
        hovertemplate=f'<b>2025 Cluster {i}</b><br>{cluster_names_2025[i]}<extra></extra>',
    ))

fig_shift.update_layout(
    **base_layout(height=750, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'],
    map_zoom=9.8,
    map_center=dict(lat=40.72, lon=-73.94),
    legend=dict(
        yanchor='top', y=0.99, xanchor='left', x=0.01,
        bgcolor='rgba(255,255,255,0.92)', bordercolor='#dde1e7',
        borderwidth=1, font=dict(size=10, family=STYLE['font_family']),
    ),
)

save_html(fig_shift, 'cluster_centroid_shifts.html')

[7.3] Cluster shifts map...
  Saved: cluster_centroid_shifts.html


---
## 8. Pickup Density Change

Zone-level change in pickup share (percentage points) between 2018 and 2025, capped at the 95th percentile of absolute values to prevent outlier zones from compressing the color range.

In [21]:
print("[8.1] Pickup density change...")

pu_counts_2018 = df_2018.groupby(['PU_zone_id', 'PU_zone_name', 'PU_borough']).size().reset_index(name='count_2018')
pu_counts_2025 = df_2025.groupby(['PU_zone_id', 'PU_zone_name', 'PU_borough']).size().reset_index(name='count_2025')
pu_counts_2018.columns = ['zone_id', 'zone_name', 'borough', 'count_2018']
pu_counts_2025.columns = ['zone_id', 'zone_name', 'borough', 'count_2025']

zone_comparison = pu_counts_2018.merge(
    pu_counts_2025, on=['zone_id', 'zone_name', 'borough'], how='outer'
).fillna(0)
zone_comparison['share_2018'] = 100 * zone_comparison['count_2018'] / zone_comparison['count_2018'].sum()
zone_comparison['share_2025'] = 100 * zone_comparison['count_2025'] / zone_comparison['count_2025'].sum()
zone_comparison['share_change'] = zone_comparison['share_2025'] - zone_comparison['share_2018']

zone_change = zone_comparison[['zone_id', 'zone_name', 'borough', 'share_change']].copy()
zone_change['zone_id'] = zone_change['zone_id'].astype(float).astype(int).astype(str)

zone_ids_change = set(zone_change['zone_id'].values)
filtered_geojson_change = {
    'type': 'FeatureCollection',
    'features': [f for f in taxi_zones_geo_4326['features']
                 if f['properties']['LocationID'] in zone_ids_change]
}

cap = zone_change['share_change'].abs().quantile(0.95)
zone_change['share_change_capped'] = zone_change['share_change'].clip(-cap, cap)

fig_pu_change = go.Figure()

fig_pu_change.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[0] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))

fig_pu_change.add_trace(go.Choroplethmap(
    geojson=filtered_geojson_change,
    locations=zone_change['zone_id'],
    featureidkey='properties.LocationID',
    z=zone_change['share_change_capped'],
    colorscale='RdBu', zmid=0,
    marker_opacity=0.85,
    marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(
        title='Change (pp)', ticksuffix=' pp', x=0.99,
        titlefont=dict(family=STYLE['font_family']),
        tickfont=dict(family=STYLE['font_family']),
    ),
    customdata=np.column_stack([
        zone_change['zone_name'], zone_change['borough'], zone_change['share_change']
    ]),
    hovertemplate=(
        '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        'Change: %{customdata[2]:.2f} pp<extra></extra>'
    ),
))

fig_pu_change.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'],
    map_zoom=STYLE['map_zoom'],
    map_center=dict(**STYLE['map_center']),
)

save_html(fig_pu_change, 'pickup_density_change.html')

[8.1] Pickup density change...


ValueError: Invalid property specified for object of type plotly.graph_objs.choroplethmap.ColorBar: 'titlefont'

Did you mean "tickfont"?

    Valid properties:
        bgcolor
            Sets the color of padded area.
        bordercolor
            Sets the axis line color.
        borderwidth
            Sets the width (in px) or the border enclosing this
            color bar.
        dtick
            Sets the step in-between ticks on this axis. Use with
            `tick0`. Must be a positive number, or special strings
            available to "log" and "date" axes. If the axis `type`
            is "log", then ticks are set every 10^(n*dtick) where n
            is the tick number. For example, to set a tick mark at
            1, 10, 100, 1000, ... set dtick to 1. To set tick marks
            at 1, 100, 10000, ... set dtick to 2. To set tick marks
            at 1, 5, 25, 125, 625, 3125, ... set dtick to
            log_10(5), or 0.69897000433. "log" has several special
     

---
## 9. Temporal Analysis

Hourly and day-of-week demand profiles compared across years, both at the aggregate level and disaggregated by cluster.

In [ ]:
# ── Hourly pickup profile ────────────────────────────────────────────────
print("[9.1] Hourly patterns...")

hourly_2018 = df_2018.groupby('hour').size()
hourly_2025 = df_2025.groupby('hour').size()
hourly_2018_pct = 100 * hourly_2018 / hourly_2018.sum()
hourly_2025_pct = 100 * hourly_2025 / hourly_2025.sum()

fig_hourly = go.Figure()
fig_hourly.add_trace(go.Scatter(
    x=hourly_2018_pct.index, y=hourly_2018_pct.values,
    name='2018', mode='lines+markers',
    line=dict(color=STYLE['year_2018'], width=2.5),
    marker=dict(size=6),
    hovertemplate='Hour %{x}: %{y:.1f}%<extra>2018</extra>',
))
fig_hourly.add_trace(go.Scatter(
    x=hourly_2025_pct.index, y=hourly_2025_pct.values,
    name='2025', mode='lines+markers',
    line=dict(color=STYLE['year_2025'], width=2.5),
    marker=dict(size=6),
    hovertemplate='Hour %{x}: %{y:.1f}%<extra>2025</extra>',
))

fig_hourly.update_layout(
    **base_layout(height=500, width=900),
    xaxis=styled_axis(title_text='Hour of Day'),
    yaxis=styled_axis(title_text='Share of Daily Trips (%)'),
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
)

save_html(fig_hourly, 'hourly_patterns.html')

In [ ]:
# ── Hourly profile by cluster (2x3 subplots) ─────────────────────────────
print("[9.2] Cluster hourly profiles...")

fig_hourly_cluster = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f"Cluster {i}: {short_labels_2018[i]}" for i in range(N_CLUSTERS)],
    shared_yaxes=True, shared_xaxes=True,
    vertical_spacing=0.12, horizontal_spacing=0.05,
)

for i in range(N_CLUSTERS):
    row = i // 3 + 1
    col = i % 3 + 1

    cluster_data_18 = df_2018[df_2018['PU_cluster'] == i]
    hourly_18 = cluster_data_18.groupby('hour').size()
    hourly_18_pct = 100 * hourly_18 / hourly_18.sum()

    matched_idx = cluster_matches[i]
    cluster_data_25 = df_2025[df_2025['PU_cluster'] == matched_idx]
    hourly_25 = cluster_data_25.groupby('hour').size()
    hourly_25_pct = 100 * hourly_25 / hourly_25.sum()

    fig_hourly_cluster.add_trace(go.Scatter(
        x=hourly_18_pct.index, y=hourly_18_pct.values,
        name='2018', mode='lines',
        line=dict(color=STYLE['year_2018'], width=2),
        showlegend=(i == 0),
        hovertemplate='Hour %{x}: %{y:.1f}%<extra>2018</extra>',
    ), row=row, col=col)

    fig_hourly_cluster.add_trace(go.Scatter(
        x=hourly_25_pct.index, y=hourly_25_pct.values,
        name='2025', mode='lines',
        line=dict(color=STYLE['year_2025'], width=2),
        showlegend=(i == 0),
        hovertemplate='Hour %{x}: %{y:.1f}%<extra>2025</extra>',
    ), row=row, col=col)

fig_hourly_cluster.update_layout(
    **base_layout(height=500, width=1100),
    legend=dict(x=0.92, y=0.98, font=dict(family=STYLE['font_family'])),
)
fig_hourly_cluster.update_xaxes(title_text='Hour', row=2)
fig_hourly_cluster.update_yaxes(title_text='% of cluster trips', col=1)

save_html(fig_hourly_cluster, 'cluster_hourly_profiles.html')

In [ ]:
# ── Demand heatmap: Hour x Day (2018 | 2025 | Diff) ─────────────────────
print("[9.3] Demand heatmaps...")

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig_heat = make_subplots(
    rows=1, cols=3,
    subplot_titles=('2018', '2025', '2025 minus 2018'),
    horizontal_spacing=0.06,
)

pivot_tables = {}
for label, df_year, col_idx in [('2018', df_2018, 1), ('2025', df_2025, 2)]:
    pivot = df_year.groupby(['day_name', 'hour']).size().reset_index(name='trips')
    pivot_table = pivot.pivot(index='day_name', columns='hour', values='trips').reindex(day_order)
    pivot_pct = 100 * pivot_table / pivot_table.sum().sum()
    pivot_tables[label] = pivot_pct

    fig_heat.add_trace(go.Heatmap(
        z=pivot_pct.values, x=pivot_pct.columns, y=pivot_pct.index,
        colorscale='Viridis', showscale=(col_idx == 2),
        colorbar=dict(
        title=dict(text='% of Total', 
        font=dict(family=STYLE['font_family'])),
        tickfont=dict(family=STYLE['font_family']),
        ticksuffix=' pp', 
        x=0.63,
        ) if col_idx == 2 else None,
        hovertemplate='<b>%{y}, Hour %{x}</b><br>Share: %{z:.2f}%<extra></extra>'
    ), row=1, col=col_idx)

diff = pivot_tables['2025'].values - pivot_tables['2018'].values
fig_heat.add_trace(go.Heatmap(
    z=diff, x=pivot_tables['2018'].columns, y=day_order,
    colorscale='RdBu', zmid=0, showscale=True,
    colorbar=dict(
    title=dict(text='Change (pp)', font=dict(family=STYLE['font_family'])),
    ticksuffix=' pp', x=0.99,
    tickfont=dict(family=STYLE['font_family']),
    ),
    hovertemplate='<b>%{y}, Hour %{x}</b><br>Change: %{z:.3f} pp<extra></extra>'
), row=1, col=3)

fig_heat.update_layout(**base_layout(height=450, width=1200))

save_html(fig_heat, 'demand_heatmaps.html')

[9.3] Demand heatmaps...
  Saved: demand_heatmaps.html


---
## 10. Borough Distribution and Spatial Concentration

Pickup counts by borough (grouped bar) and Lorenz curves measuring how concentrated demand is across zones, with Gini coefficients quantifying the degree of inequality.

In [ ]:
# ── Borough pickup distribution ──────────────────────────────────────────
print("[10.1] Borough analysis...")

borough_df_2018 = df_2018.groupby('PU_borough').size().reset_index(name='trips_2018')
borough_df_2025 = df_2025.groupby('PU_borough').size().reset_index(name='trips_2025')
borough_df = borough_df_2018.merge(borough_df_2025, on='PU_borough')

fig_borough = go.Figure()
fig_borough.add_trace(go.Bar(
    x=borough_df['PU_borough'], y=borough_df['trips_2018'],
    name='2018', marker_color=STYLE['year_2018'],
))
fig_borough.add_trace(go.Bar(
    x=borough_df['PU_borough'], y=borough_df['trips_2025'],
    name='2025', marker_color=STYLE['year_2025'],
))

fig_borough.update_layout(
    **base_layout(height=500, width=900),
    xaxis=styled_axis(title_text='Borough'),
    yaxis=styled_axis(title_text='Number of Trips'),
    barmode='group',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])),
)

save_html(fig_borough, 'borough_analysis.html')

[10.1] Borough analysis...
  Saved: borough_analysis.html


In [ ]:
# ── Lorenz curve ─────────────────────────────────────────────────────────
print("[10.2] Lorenz curve...")

cz_18, ct_18 = lorenz_data(df_2018, 'PU_zone_id')
cz_25, ct_25 = lorenz_data(df_2025, 'PU_zone_id')
gini_18 = gini_from_lorenz(cz_18, ct_18)
gini_25 = gini_from_lorenz(cz_25, ct_25)

fig_lorenz = go.Figure()
fig_lorenz.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(color='#888', width=1.5, dash='dash'),
    name='Perfect equality', hoverinfo='skip'
))
fig_lorenz.add_trace(go.Scatter(
    x=np.concatenate([[0], cz_18]), y=np.concatenate([[0], ct_18]),
    mode='lines', line=dict(color=STYLE['year_2018'], width=2.5),
    name=f'2018 (Gini = {gini_18:.3f})',
    hovertemplate='%{x:.0%} of zones -> %{y:.0%} of trips<extra>2018</extra>'
))
fig_lorenz.add_trace(go.Scatter(
    x=np.concatenate([[0], cz_25]), y=np.concatenate([[0], ct_25]),
    mode='lines', line=dict(color=STYLE['year_2025'], width=2.5),
    name=f'2025 (Gini = {gini_25:.3f})',
    hovertemplate='%{x:.0%} of zones -> %{y:.0%} of trips<extra>2025</extra>'
))

fig_lorenz.update_layout(
    **base_layout(height=600, width=850),
    xaxis=styled_axis(title_text='Cumulative Share of Zones', range=[0, 1], tickformat='.0%'),
    yaxis=styled_axis(title_text='Cumulative Share of Trips', range=[0, 1], tickformat='.0%'),
    legend=dict(x=0.05, y=0.95, font=dict(family=STYLE['font_family'])),
)

save_html(fig_lorenz, 'lorenz_curve.html')

[10.2] Lorenz curve...
  Saved: lorenz_curve.html


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_949/1079649660.py:48: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



---
## 11. Intra-Cluster Trip Share

Fraction of trips where pickup and dropoff fall within the same geographic cluster, indicating the degree of within-area versus cross-area travel.

In [ ]:
print("[11.1] Intra-cluster share...")

intra_2018 = df_2018.dropna(subset=['PU_cluster', 'DO_cluster'])
intra_2018_count = (intra_2018['PU_cluster'] == intra_2018['DO_cluster']).sum()
intra_2018_pct = 100 * intra_2018_count / len(intra_2018)

intra_2025 = df_2025.dropna(subset=['PU_cluster', 'DO_cluster'])
intra_2025_count = (intra_2025['PU_cluster'] == intra_2025['DO_cluster']).sum()
intra_2025_pct = 100 * intra_2025_count / len(intra_2025)

fig_intra = go.Figure()
fig_intra.add_trace(go.Bar(
    x=['2018', '2025'],
    y=[intra_2018_pct, intra_2025_pct],
    marker_color=[STYLE['year_2018'], STYLE['year_2025']],
    text=[f'{intra_2018_pct:.1f}%', f'{intra_2025_pct:.1f}%'],
    textposition='outside',
    width=0.4,
))

fig_intra.update_layout(
    **base_layout(height=500, width=900, margin=dict(l=80, r=80, t=20, b=50)),
    xaxis=styled_axis(range=[-0.5, 1.5]),
    yaxis=styled_axis(title_text='% of Trips (Same Cluster PU and DO)'),
    showlegend=False,
)

save_html(fig_intra, 'intra_cluster_share.html')

[11.1] Intra-cluster share...
  Saved: intra_cluster_share.html


---
## 12. Spatial Autocorrelation: LISA Maps

Local Indicators of Spatial Association (Moran's Local I) identify statistically significant clusters and outliers in the distribution of trip share across taxi zones. Hot spots (High-High) indicate zones with high demand surrounded by high-demand neighbors; cold spots (Low-Low) indicate the opposite. The spatial weights matrix uses K=6 nearest neighbors with row standardization. Global Moran's I is reported as an annotation.

In [ ]:
# ── Pickup/dropoff mismatch statistics ───────────────────────────────────
print("[12.0] Mismatch ratio statistics...")

mr_2018 = mismatch_ratios(df_2018)
mr_2025 = mismatch_ratios(df_2025)

ks_stat, ks_p = ks_2samp(mr_2018['ratio'].dropna(), mr_2025['ratio'].dropna())
lev_stat, lev_p = levene(mr_2018['ratio'].dropna(), mr_2025['ratio'].dropna())

print(f"  KS test: stat={ks_stat:.4f}, p={ks_p:.4e}")
print(f"  Levene test: stat={lev_stat:.4f}, p={lev_p:.4e}")

[12.0] Mismatch ratio statistics...
  KS test: stat=0.2085, p=2.4268e-05
  Levene test: stat=0.0033, p=9.5430e-01


In [ ]:
# ── LISA cluster maps ────────────────────────────────────────────────────
print("[12.1] LISA cluster maps...")

w = libpysal.weights.KNN.from_dataframe(gdf_raw, k=6)
w.transform = 'r'

for year_label, df_year in [('2018', df_2018), ('2025', df_2025)]:
    zone_counts = df_year.groupby('PU_zone_id').size()
    gdf_tmp = gdf_raw.copy()
    gdf_tmp['LocationID_int'] = gdf_tmp['LocationID'].astype(int)
    gdf_tmp['trips'] = gdf_tmp['LocationID_int'].map(zone_counts).fillna(0)
    gdf_tmp['trip_share'] = 100 * gdf_tmp['trips'] / gdf_tmp['trips'].sum()

    lisa = Moran_Local(gdf_tmp['trip_share'].values, w, permutations=999)
    mi_global = Moran(gdf_tmp['trip_share'].values, w)

    sig = lisa.p_sim < 0.05
    quadrant_map = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
    gdf_tmp['lisa_class'] = 'ns'
    for idx in range(len(gdf_tmp)):
        if sig[idx]:
            gdf_tmp.iloc[idx, gdf_tmp.columns.get_loc('lisa_class')] = \
                quadrant_map.get(lisa.q[idx], 'ns')

    gdf_tmp['zone_id_str'] = gdf_tmp['LocationID_int'].astype(str)
    gdf_tmp['color'] = gdf_tmp['lisa_class'].map(STYLE['lisa_colors'])

    zc_lookup = zone_centroids.drop_duplicates(subset='zone_id').set_index('zone_id')
    gdf_tmp['zone_name'] = gdf_tmp['LocationID_int'].map(zc_lookup['zone_name'])
    gdf_tmp['borough_name'] = gdf_tmp['LocationID_int'].map(zc_lookup['borough'])

    geojson_all = json.loads(gdf_tmp.to_json())
    for feat, zid, lclass in zip(
        geojson_all['features'],
        gdf_tmp['zone_id_str'],
        gdf_tmp['lisa_class']
    ):
        feat['properties']['zone_id_str'] = zid
        feat['properties']['lisa_class'] = lclass

    fig_lisa = go.Figure()

    fig_lisa.add_trace(go.Choroplethmap(
        geojson=taxi_zones_geo_4326,
        locations=all_zone_ids,
        featureidkey='properties.LocationID',
        z=[1] * len(all_zone_ids),
        colorscale=[[0, '#f5f5f5'], [1, '#f5f5f5']],
        marker_opacity=0.3, marker_line_width=0.3, marker_line_color='#ccc',
        showscale=False, hoverinfo='skip',
    ))

    for lclass in ['HH', 'LL', 'HL', 'LH', 'ns']:
        subset = gdf_tmp[gdf_tmp['lisa_class'] == lclass]
        if len(subset) == 0:
            continue

        subset_geojson = {
            'type': 'FeatureCollection',
            'features': [f for f in geojson_all['features']
                         if f['properties']['lisa_class'] == lclass]
        }

        fig_lisa.add_trace(go.Choroplethmap(
            geojson=subset_geojson,
            locations=subset['zone_id_str'].values,
            featureidkey='properties.zone_id_str',
            z=[1] * len(subset),
            colorscale=[[0, STYLE['lisa_colors'][lclass]], [1, STYLE['lisa_colors'][lclass]]],
            marker_opacity=0.8 if lclass != 'ns' else 0.35,
            marker_line_width=0.5,
            marker_line_color='rgba(255,255,255,0.6)',
            showscale=False,
            name=STYLE['lisa_labels'][lclass],
            showlegend=True,
            customdata=np.column_stack([
                subset['zone_name'].fillna('Unknown').values,
                subset['borough_name'].fillna('Unknown').values,
                subset['trip_share'].values,
            ]),
            hovertemplate=(
                '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
                'Trip share: %{customdata[2]:.2f}%<br>'
                f'LISA: {STYLE["lisa_labels"][lclass]}'
                '<extra></extra>'
            ),
        ))

    fig_lisa.update_layout(
        **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
        map_style=STYLE['map_style'],
        map_zoom=STYLE['map_zoom'],
        map_center=dict(**STYLE['map_center']),
        legend=dict(
            title="LISA Classification",
            yanchor='top', y=0.99, xanchor='left', x=0.01,
            bgcolor='rgba(255,255,255,0.95)', bordercolor='#dde1e7',
            borderwidth=1, font=dict(size=STYLE['legend_size'], family=STYLE['font_family']),
        ),
        annotations=[dict(
            x=0.99, y=0.01, xref='paper', yref='paper',
            xanchor='right', yanchor='bottom',
            text=f"Global Moran's I: {mi_global.I:.4f} (p={mi_global.p_sim:.4f})",
            showarrow=False, font=dict(size=11, family=STYLE['font_family']),
            bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1,
        )]
    )

    save_html(fig_lisa, f'lisa_map_{year_label}.html')

[12.1] LISA cluster maps...
  Saved: lisa_map_2018.html
  Saved: lisa_map_2025.html


---
## 13. Summary

In [ ]:
matched_shifts = [
    haversine_km(
        centers_2018[i18, 0], centers_2018[i18, 1],
        centers_2025[i25, 0], centers_2025[i25, 1]
    )
    for i18, i25 in cluster_matches.items()
]
avg_shift = np.mean(matched_shifts)

print("=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print(f"  Market share: {100 * uber_count_2018 / total_2018:.1f}% -> {100 * uber_count_2025 / total_2025:.1f}%")
print(f"  Gini: {gini_18:.3f} -> {gini_25:.3f}")
print(f"  Avg cluster shift: {avg_shift:.2f} km")
print(f"  Intra-cluster trips: {intra_2018_pct:.1f}% -> {intra_2025_pct:.1f}%")
print(f"  KS mismatch test: stat={ks_stat:.4f}, p={ks_p:.4e}")
print(f"  Levene mismatch test: stat={lev_stat:.4f}, p={lev_p:.4e}")
print("=" * 70)

print(f"\nOutputs saved to: {OUTPUT_DIR}")
print("  Charts:")
for fname in [
    'cluster_map_2018.html',
    'cluster_map_2025.html',
    'cluster_centroid_shifts.html',
    'pickup_density_change.html',
    'hourly_patterns.html',
    'cluster_hourly_profiles.html',
    'demand_heatmaps.html',
    'borough_analysis.html',
    'lorenz_curve.html',
    'intra_cluster_share.html',
    'lisa_map_2018.html',
    'lisa_map_2025.html',
]:
    print(f"    {fname}")

ANALYSIS COMPLETE
  Market share: 22.7% -> 75.3%
  Gini: 0.511 -> 0.440
  Avg cluster shift: 4.22 km
  Intra-cluster trips: 63.2% -> 71.9%
  KS mismatch test: stat=0.2085, p=2.4268e-05
  Levene mismatch test: stat=0.0033, p=9.5430e-01

Outputs saved to: /Users/leoss/Desktop/Portfolio/Website-/projects/uber/outputs/
  Charts:
    cluster_map_2018.html
    cluster_map_2025.html
    cluster_centroid_shifts.html
    pickup_density_change.html
    hourly_patterns.html
    cluster_hourly_profiles.html
    demand_heatmaps.html
    borough_analysis.html
    lorenz_curve.html
    intra_cluster_share.html
    lisa_map_2018.html
    lisa_map_2025.html


In [ ]:
"""
Uber NYC Analysis - Extensions Part 1 (Suggestions 1-5)
========================================================
Standalone script that adds five new analyses to the original Uber NYC project.

1. Confounders with data: Uber vs Lyft vs other FHV market share breakdown
2. Multi-year time series: Gini and Moran's I across intermediate January snapshots
3. Dropoff / OD flow analysis: cluster-to-cluster Sankey + dropoff density change
4. Trip distance analysis: haversine PU-DO distances by year and cluster
5. Demand-feature clustering: cluster zones by hourly demand profile, not coordinates

Requirements:
    - Original notebook paths and data files (Jan 2018, Jan 2025 parquets)
    - For suggestion 2, place intermediate January parquet files in DATA_DIR:
        fhv_tripdata_2019-01.parquet
        fhv_tripdata_2020-01.parquet
        fhvhv_tripdata_2021-01.parquet  (schema changed in mid-2019)
        fhvhv_tripdata_2022-01.parquet
        fhvhv_tripdata_2023-01.parquet
        fhvhv_tripdata_2024-01.parquet
      Download from: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
      The script will skip missing files and use whatever is available.

Outputs saved to OUTPUT_DIR as interactive Plotly HTML files.

Methods:
    - Market share: filter by dispatching_base_num (2018) or hvfhs_license_num (2019+)
    - Gini coefficient: trapezoidal integration under Lorenz curve of zone pickup shares
    - Moran's I: global spatial autocorrelation via PySAL (KNN k=6, row-standardized)
    - OD flows: cluster-to-cluster trip counts normalized to shares, rendered as Sankey
    - Trip distance: vectorized Haversine great-circle km between PU and DO zone centroids
    - Demand clustering: K-means on 24-dim hourly pickup profile vectors per zone
"""

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
from math import radians, cos, sin, asin, sqrt
import geopandas as gpd
import libpysal
from esda.moran import Moran
import json
import gc
import os
import glob

# ==========================================================================
# CONFIGURATION (matches original notebook)
# ==========================================================================
BASE_DIR = '/Users/leoss/Desktop/Portfolio/Website-/'
DATA_DIR = BASE_DIR + 'projects/uber/data/'
OUTPUT_DIR = BASE_DIR + 'projects/uber/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_2018 = DATA_DIR + 'fhv_tripdata_2018-01.parquet'
PATH_2025 = DATA_DIR + 'fhvhv_tripdata_2025-01.parquet'
PATH_CENTROIDS = DATA_DIR + 'zone_centroids.csv'
PATH_SHAPEFILE = DATA_DIR + 'taxi_zones/taxi_zones.shp'

SAMPLE_SIZE = 20_000_000
N_CLUSTERS = 6

UBER_2018_BASES = ['B02512', 'B02598', 'B02617', 'B02682', 'B02764', 'B02765', 'B02835', 'B02836']
LYFT_2018_BASES = ['B02510']
UBER_2025_LICENSE = 'HV0003'
LYFT_2025_LICENSE = 'HV0005'

AIRPORT_ZONE_IDS = {132, 138, 1}

# Style system (from original)
STYLE = {
    'font_family': 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size': 11,
    'axis_title_size': 13,
    'legend_size': 11,
    'template': 'plotly_white',
    'plot_bg': 'rgba(0,0,0,0)',
    'paper_bg': 'white',
    'chart_height': 550,
    'margin_default': dict(l=60, r=40, t=20, b=50),
    'margin_map': dict(l=0, r=0, t=20, b=0),
    'grid_color': '#e5e7eb',
    'grid_width': 0.5,
    'hover_font_size': 13,
    'hover_font_color': '#1a2744',
    'year_2018': '#ff6b6b',
    'year_2025': '#4ecdc4',
    'cluster_colors': ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4'],
    'map_style': 'carto-positron-nolabels',
    'map_center': {'lat': 40.7128, 'lon': -73.9352},
    'map_zoom': 9.5,
    'lisa_colors': {
        'HH': '#d7191c', 'LL': '#2c7bb6', 'HL': '#fdae61',
        'LH': '#abd9e9', 'ns': '#e8e8e8',
    },
}


def base_layout(height=None, width=None, **kwargs):
    layout = dict(
        title='',
        font=dict(family=STYLE['font_family']),
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        height=height or STYLE['chart_height'],
        margin=STYLE['margin_default'],
        hoverlabel=dict(
            font_size=STYLE['hover_font_size'],
            font_color=STYLE['hover_font_color'],
        ),
    )
    if width:
        layout['width'] = width
    layout.update(kwargs)
    return layout


def styled_axis(**kwargs):
    return dict(
        tickfont=dict(size=STYLE['tick_size']),
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'],
        gridwidth=STYLE['grid_width'],
        **kwargs,
    )


def save_html(fig, filename):
    fig.write_html(
        OUTPUT_DIR + filename,
        include_plotlyjs='cdn',
        config={'displayModeBar': False},
    )
    print(f"  Saved: {filename}")


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * 6371 * asin(sqrt(a))


def lorenz_data(zone_counts_series):
    """Lorenz curve from a Series of zone-level counts."""
    vals = zone_counts_series.sort_values().values
    cum_zones = np.arange(1, len(vals) + 1) / len(vals)
    cum_trips = np.cumsum(vals) / vals.sum()
    return cum_zones, cum_trips


def gini_from_lorenz(cum_zones, cum_trips):
    return 1 - 2 * np.trapz(cum_trips, cum_zones)


def name_cluster_by_location(center_lat, center_lon, zone_centroids):
    zone_distances = np.sqrt(
        (zone_centroids['latitude'] - center_lat) ** 2 +
        (zone_centroids['longitude'] - center_lon) ** 2
    )
    nearest_idx = zone_distances.idxmin()
    nearest_zone = zone_centroids.iloc[nearest_idx]
    return f"{nearest_zone['borough']}: {nearest_zone['zone_name']}"


def match_clusters_by_proximity(centers_a, centers_b):
    distances = cdist(centers_a, centers_b, metric='cityblock')
    matches = {}
    used_b = set()
    pairs = []
    for i in range(len(centers_a)):
        for j in range(len(centers_b)):
            pairs.append((i, j, distances[i, j]))
    pairs.sort(key=lambda x: x[2])
    for i, j, dist in pairs:
        if i not in matches and j not in used_b:
            matches[i] = j
            used_b.add(j)
    return matches


def merge_zone_info(df, zone_col, centroids, prefix):
    merged = df.merge(
        centroids[['zone_id', 'zone_name', 'borough', 'latitude', 'longitude']],
        left_on=zone_col, right_on='zone_id', how='left'
    )
    merged = merged.rename(columns={
        'zone_id': f'{prefix}_zone_id',
        'zone_name': f'{prefix}_zone_name',
        'borough': f'{prefix}_borough',
        'latitude': f'{prefix}_lat',
        'longitude': f'{prefix}_lon',
    })
    return merged


# ==========================================================================
# LOAD COMMON DATA
# ==========================================================================
print("=" * 70)
print("LOADING COMMON DATA")
print("=" * 70)

zone_centroids = pd.read_csv(PATH_CENTROIDS)
print(f"Loaded {len(zone_centroids)} zone centroids")

gdf_raw = gpd.read_file(PATH_SHAPEFILE)
gdf_raw = gdf_raw.to_crs(epsg=4326)
taxi_zones_geo_4326 = json.loads(gdf_raw.to_json())
for f in taxi_zones_geo_4326['features']:
    f['properties']['LocationID'] = str(int(f['properties']['LocationID']))
all_zone_ids = [f['properties']['LocationID'] for f in taxi_zones_geo_4326['features']]


# ==========================================================================
# LOAD 2018 + 2025 DATA (same as original notebook)
# ==========================================================================
def load_and_process_year(path, year, uber_filter_col, uber_filter_val, 
                          pu_col, do_col, sample_size=SAMPLE_SIZE):
    """Load, filter to Uber, sample, merge zone info, cluster."""
    print(f"\n[Loading {year}] {path}")
    
    table = pq.read_table(path, columns=[])
    total = table.num_rows
    print(f"  Total rows: {total:,}")
    
    if year <= 2018:
        cols = ['pickup_datetime', pu_col, do_col, uber_filter_col]
    else:
        cols = ['pickup_datetime', pu_col, do_col, uber_filter_col]
    
    table = pq.read_table(path, columns=cols)
    df = table.to_pandas()
    
    if isinstance(uber_filter_val, list):
        df_uber = df[df[uber_filter_col].isin(uber_filter_val)].copy()
    else:
        df_uber = df[df[uber_filter_col] == uber_filter_val].copy()
    
    uber_count = len(df_uber)
    print(f"  Uber trips: {uber_count:,} ({100 * uber_count / total:.1f}%)")
    
    if len(df_uber) > sample_size:
        df_uber = df_uber.sample(n=sample_size, random_state=42)
    
    df_uber['pickup_datetime'] = pd.to_datetime(df_uber['pickup_datetime'])
    df_uber['hour'] = df_uber['pickup_datetime'].dt.hour
    df_uber['day_of_week'] = df_uber['pickup_datetime'].dt.dayofweek
    df_uber['day_name'] = df_uber['pickup_datetime'].dt.day_name()
    
    df_uber = df_uber.dropna(subset=[pu_col])
    df_uber[pu_col] = df_uber[pu_col].astype(int)
    df_uber = merge_zone_info(df_uber, pu_col, zone_centroids, 'PU')
    df_uber = df_uber.dropna(subset=['PU_lat', 'PU_lon'])
    
    df_uber[do_col] = pd.to_numeric(df_uber[do_col], errors='coerce')
    df_uber.loc[df_uber[do_col].notna(), do_col] = \
        df_uber.loc[df_uber[do_col].notna(), do_col].astype(int)
    df_uber = merge_zone_info(df_uber, do_col, zone_centroids, 'DO')
    
    # Cluster
    coords = df_uber[['PU_lat', 'PU_lon']].values
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    df_uber['PU_cluster'] = kmeans.fit_predict(coords)
    
    cluster_names = {}
    for i in range(N_CLUSTERS):
        lat, lon = kmeans.cluster_centers_[i]
        cluster_names[i] = name_cluster_by_location(lat, lon, zone_centroids)
    df_uber['PU_cluster_name'] = df_uber['PU_cluster'].map(cluster_names)
    
    do_mask = df_uber['DO_lat'].notna() & df_uber['DO_lon'].notna()
    df_uber.loc[do_mask, 'DO_cluster'] = kmeans.predict(
        df_uber.loc[do_mask, ['DO_lat', 'DO_lon']].values
    )
    df_uber['DO_cluster_name'] = df_uber['DO_cluster'].map(cluster_names)
    
    del table, df
    gc.collect()
    
    return df_uber, kmeans, cluster_names, total, uber_count


print("\n" + "=" * 70)
print("LOADING MAIN DATASETS")
print("=" * 70)

df_2018, kmeans_2018, cnames_2018, total_2018, uber_n_2018 = load_and_process_year(
    PATH_2018, 2018, 'dispatching_base_num', UBER_2018_BASES,
    'PUlocationID', 'DOlocationID'
)

df_2025, kmeans_2025, cnames_2025, total_2025, uber_n_2025 = load_and_process_year(
    PATH_2025, 2025, 'hvfhs_license_num', UBER_2025_LICENSE,
    'PULocationID', 'DOLocationID'
)

centers_2018 = kmeans_2018.cluster_centers_
centers_2025 = kmeans_2025.cluster_centers_
cluster_matches = match_clusters_by_proximity(centers_2018, centers_2025)


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 1: CONFOUNDERS - MARKET SHARE BREAKDOWN
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[1] CONFOUNDER ANALYSIS: MARKET SHARE BREAKDOWN")
print("=" * 70)

# -- 2018: use dispatching_base_num --
print("[1.1] 2018 market share breakdown...")
table_ms_2018 = pq.read_table(PATH_2018, columns=['dispatching_base_num'])
df_ms_2018 = table_ms_2018.to_pandas()
total_trips_2018 = len(df_ms_2018)

uber_mask_2018 = df_ms_2018['dispatching_base_num'].isin(UBER_2018_BASES)
lyft_mask_2018 = df_ms_2018['dispatching_base_num'].isin(LYFT_2018_BASES)

n_uber_2018 = uber_mask_2018.sum()
n_lyft_2018 = lyft_mask_2018.sum()
n_other_2018 = total_trips_2018 - n_uber_2018 - n_lyft_2018

del table_ms_2018, df_ms_2018
gc.collect()

# -- 2025: use hvfhs_license_num --
print("[1.2] 2025 market share breakdown...")
table_ms_2025 = pq.read_table(PATH_2025, columns=['hvfhs_license_num'])
df_ms_2025 = table_ms_2025.to_pandas()
total_trips_2025 = len(df_ms_2025)

n_uber_2025 = (df_ms_2025['hvfhs_license_num'] == UBER_2025_LICENSE).sum()
n_lyft_2025 = (df_ms_2025['hvfhs_license_num'] == LYFT_2025_LICENSE).sum()
n_other_2025 = total_trips_2025 - n_uber_2025 - n_lyft_2025

del table_ms_2025, df_ms_2025
gc.collect()

# Build comparison dataframe
ms_data = pd.DataFrame({
    'Operator': ['Uber', 'Lyft', 'Other FHV'] * 2,
    'Year': ['2018'] * 3 + ['2025'] * 3,
    'Trips': [n_uber_2018, n_lyft_2018, n_other_2018,
              n_uber_2025, n_lyft_2025, n_other_2025],
    'Share': [
        100 * n_uber_2018 / total_trips_2018,
        100 * n_lyft_2018 / total_trips_2018,
        100 * n_other_2018 / total_trips_2018,
        100 * n_uber_2025 / total_trips_2025,
        100 * n_lyft_2025 / total_trips_2025,
        100 * n_other_2025 / total_trips_2025,
    ]
})

print("\n  Market share breakdown:")
for _, row in ms_data.iterrows():
    print(f"    {row['Year']} {row['Operator']}: {row['Trips']:>12,} ({row['Share']:.1f}%)")

# -- Grouped bar chart --
operator_colors = {'Uber': '#1a1a2e', 'Lyft': '#ff00bf', 'Other FHV': '#888888'}

fig_ms = go.Figure()
for op in ['Uber', 'Lyft', 'Other FHV']:
    subset = ms_data[ms_data['Operator'] == op]
    fig_ms.add_trace(go.Bar(
        x=subset['Year'], y=subset['Share'],
        name=op, marker_color=operator_colors[op],
        text=[f"{s:.1f}%" for s in subset['Share']],
        textposition='outside',
        hovertemplate=(
            f'<b>{op}</b><br>'
            'Year: %{x}<br>'
            'Share: %{y:.1f}%<br>'
            f'Trips: %{{customdata:,}}<extra></extra>'
        ),
        customdata=subset['Trips'].values,
    ))

fig_ms.update_layout(
    **base_layout(height=500, width=850),
    xaxis=styled_axis(title_text=''),
    yaxis=styled_axis(title_text='Market Share (%)', range=[0, 85]),
    barmode='group',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
)

save_html(fig_ms, 'ext1_market_share_breakdown.html')

# -- Stacked percentage bar (absolute + share) --
fig_ms_stack = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Absolute Trip Counts', 'Market Share (%)'),
    horizontal_spacing=0.12,
)

for op in ['Uber', 'Lyft', 'Other FHV']:
    subset = ms_data[ms_data['Operator'] == op]
    fig_ms_stack.add_trace(go.Bar(
        x=subset['Year'], y=subset['Trips'],
        name=op, marker_color=operator_colors[op],
        legendgroup=op, showlegend=True,
        hovertemplate=f'<b>{op}</b><br>Trips: %{{y:,}}<extra></extra>',
    ), row=1, col=1)
    fig_ms_stack.add_trace(go.Bar(
        x=subset['Year'], y=subset['Share'],
        name=op, marker_color=operator_colors[op],
        legendgroup=op, showlegend=False,
        hovertemplate=f'<b>{op}</b><br>Share: %{{y:.1f}}%<extra></extra>',
    ), row=1, col=2)

fig_ms_stack.update_layout(
    **base_layout(height=500, width=1100),
    barmode='stack',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])),
)
fig_ms_stack.update_yaxes(title_text='Total Trips', col=1)
fig_ms_stack.update_yaxes(title_text='Share (%)', col=2)

save_html(fig_ms_stack, 'ext1_market_share_stacked.html')


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 2: MULTI-YEAR TIME SERIES (Gini + Moran's I)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[2] MULTI-YEAR TIME SERIES: GINI AND MORAN'S I")
print("=" * 70)

# Scan for available January parquet files
# TLC changed the schema in mid-2019:
#   Pre-2019: fhv_tripdata_YYYY-01.parquet with dispatching_base_num
#   2019+: fhvhv_tripdata_YYYY-01.parquet with hvfhs_license_num
# We handle both schemas.

year_configs = {
    2018: {
        'path': DATA_DIR + 'fhv_tripdata_2018-01.parquet',
        'uber_col': 'dispatching_base_num',
        'uber_val': UBER_2018_BASES,
        'pu_col': 'PUlocationID',
    },
    2019: {
        'path': DATA_DIR + 'fhv_tripdata_2019-01.parquet',
        'uber_col': 'dispatching_base_num',
        'uber_val': UBER_2018_BASES,
        'pu_col': 'PUlocationID',
    },
    2020: {
        'path': DATA_DIR + 'fhvhv_tripdata_2020-01.parquet',
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    },
    2021: {
        'path': DATA_DIR + 'fhvhv_tripdata_2021-01.parquet',
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    },
    2022: {
        'path': DATA_DIR + 'fhvhv_tripdata_2022-01.parquet',
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    },
    2023: {
        'path': DATA_DIR + 'fhvhv_tripdata_2023-01.parquet',
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    },
    2024: {
        'path': DATA_DIR + 'fhvhv_tripdata_2024-01.parquet',
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    },
    2025: {
        'path': DATA_DIR + 'fhvhv_tripdata_2025-01.parquet',
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    },
}

# Also try alternate path patterns for 2019 (could be fhvhv)
alt_2019 = DATA_DIR + 'fhvhv_tripdata_2019-01.parquet'
if os.path.exists(alt_2019) and not os.path.exists(year_configs[2019]['path']):
    year_configs[2019] = {
        'path': alt_2019,
        'uber_col': 'hvfhs_license_num',
        'uber_val': UBER_2025_LICENSE,
        'pu_col': 'PULocationID',
    }

# Prepare spatial weights (once)
w = libpysal.weights.KNN.from_dataframe(gdf_raw, k=6)
w.transform = 'r'
gdf_base = gdf_raw.copy()
gdf_base['LocationID_int'] = gdf_base['LocationID'].astype(int)

time_series_results = []

for year, cfg in sorted(year_configs.items()):
    if not os.path.exists(cfg['path']):
        print(f"  [SKIP] {year}: file not found ({os.path.basename(cfg['path'])})")
        continue
    
    print(f"  [Processing {year}]...")
    try:
        cols = [cfg['pu_col'], cfg['uber_col']]
        table = pq.read_table(cfg['path'], columns=cols)
        df_tmp = table.to_pandas()
        total = len(df_tmp)
        
        # Filter to Uber
        if isinstance(cfg['uber_val'], list):
            df_tmp = df_tmp[df_tmp[cfg['uber_col']].isin(cfg['uber_val'])].copy()
        else:
            df_tmp = df_tmp[df_tmp[cfg['uber_col']] == cfg['uber_val']].copy()
        
        uber_n = len(df_tmp)
        uber_share = 100 * uber_n / total
        
        # Zone counts
        df_tmp = df_tmp.dropna(subset=[cfg['pu_col']])
        df_tmp[cfg['pu_col']] = df_tmp[cfg['pu_col']].astype(int)
        zone_counts = df_tmp.groupby(cfg['pu_col']).size()
        
        # Gini
        cz, ct = lorenz_data(zone_counts)
        gini = gini_from_lorenz(cz, ct)
        
        # Moran's I
        gdf_tmp = gdf_base.copy()
        gdf_tmp['trips'] = gdf_tmp['LocationID_int'].map(zone_counts).fillna(0)
        gdf_tmp['trip_share'] = 100 * gdf_tmp['trips'] / gdf_tmp['trips'].sum()
        mi = Moran(gdf_tmp['trip_share'].values, w)
        
        time_series_results.append({
            'year': year,
            'total_fhv': total,
            'uber_trips': uber_n,
            'uber_share': uber_share,
            'gini': gini,
            'morans_i': mi.I,
            'morans_p': mi.p_sim,
        })
        
        print(f"    Uber share: {uber_share:.1f}%, Gini: {gini:.3f}, "
              f"Moran's I: {mi.I:.3f} (p={mi.p_sim:.4f})")
        
        del table, df_tmp, gdf_tmp
        gc.collect()
        
    except Exception as e:
        print(f"    ERROR: {e}")
        continue

ts_df = pd.DataFrame(time_series_results)

if len(ts_df) >= 2:
    # -- Time series plot: Gini + Moran's I --
    fig_ts = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Gini Coefficient (Pickup Concentration)",
                        "Global Moran's I (Spatial Autocorrelation)"),
        horizontal_spacing=0.12,
    )
    
    fig_ts.add_trace(go.Scatter(
        x=ts_df['year'], y=ts_df['gini'],
        mode='lines+markers', name='Gini',
        line=dict(color='#4363d8', width=2.5),
        marker=dict(size=10, color='#4363d8'),
        hovertemplate='%{x}: Gini = %{y:.3f}<extra></extra>',
    ), row=1, col=1)
    
    fig_ts.add_trace(go.Scatter(
        x=ts_df['year'], y=ts_df['morans_i'],
        mode='lines+markers', name="Moran's I",
        line=dict(color='#e6194b', width=2.5),
        marker=dict(size=10, color='#e6194b'),
        hovertemplate="%{x}: Moran's I = %{y:.3f}<extra></extra>",
    ), row=1, col=2)
    
    fig_ts.update_layout(
        **base_layout(height=450, width=1100),
        showlegend=False,
    )
    fig_ts.update_xaxes(
        tickmode='array', tickvals=ts_df['year'].tolist(),
        title_text='January of Each Year'
    )
    fig_ts.update_yaxes(title_text='Gini Coefficient', col=1)
    fig_ts.update_yaxes(title_text="Moran's I", col=2)
    
    save_html(fig_ts, 'ext2_timeseries_gini_moran.html')
    
    # -- Uber market share time series --
    fig_share_ts = go.Figure()
    fig_share_ts.add_trace(go.Scatter(
        x=ts_df['year'], y=ts_df['uber_share'],
        mode='lines+markers',
        line=dict(color='#1a1a2e', width=2.5),
        marker=dict(size=10),
        text=[f"{s:.1f}%" for s in ts_df['uber_share']],
        textposition='top center',
        hovertemplate='%{x}: %{y:.1f}%<extra>Uber Share</extra>',
    ))
    fig_share_ts.update_layout(
        **base_layout(height=400, width=800),
        xaxis=styled_axis(
            title_text='January of Each Year',
            tickmode='array', tickvals=ts_df['year'].tolist(),
        ),
        yaxis=styled_axis(title_text='Uber Market Share (%)', range=[0, 100]),
        showlegend=False,
    )
    save_html(fig_share_ts, 'ext2_uber_share_timeseries.html')

else:
    print("  Only 2 data points available. Skipping time series plots.")
    print("  To enable, download intermediate January parquet files from TLC.")
    print("  See script docstring for details.")


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 3: DROPOFF / OD FLOW ANALYSIS
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[3] DROPOFF AND OD FLOW ANALYSIS")
print("=" * 70)

def short_label(name):
    return name.split(':')[0].strip()

# -- 3a. Cluster-to-cluster OD flow Sankey --
def build_od_matrix(df, n_clusters, cluster_names, label=''):
    """Build cluster-to-cluster OD share matrix."""
    valid = df.dropna(subset=['PU_cluster', 'DO_cluster']).copy()
    valid['PU_cluster'] = valid['PU_cluster'].astype(int)
    valid['DO_cluster'] = valid['DO_cluster'].astype(int)
    
    od = valid.groupby(['PU_cluster', 'DO_cluster']).size().reset_index(name='trips')
    od['share'] = 100 * od['trips'] / od['trips'].sum()
    
    print(f"  [{label}] OD matrix: {len(valid):,} valid trips, "
          f"{len(od)} cluster pairs")
    return od


od_2018 = build_od_matrix(df_2018, N_CLUSTERS, cnames_2018, '2018')
od_2025 = build_od_matrix(df_2025, N_CLUSTERS, cnames_2025, '2025')

# Sankey for each year
for year_label, od_df, c_names in [('2018', od_2018, cnames_2018), 
                                     ('2025', od_2025, cnames_2025)]:
    labels_pu = [f"PU: {short_label(c_names[i])}" for i in range(N_CLUSTERS)]
    labels_do = [f"DO: {short_label(c_names[i])}" for i in range(N_CLUSTERS)]
    all_labels = labels_pu + labels_do
    
    sources = od_df['PU_cluster'].astype(int).tolist()
    targets = (od_df['DO_cluster'].astype(int) + N_CLUSTERS).tolist()
    values = od_df['trips'].tolist()
    
    # Colors
    node_colors = [STYLE['cluster_colors'][i % len(STYLE['cluster_colors'])]
                   for i in range(N_CLUSTERS)] * 2
    
    def hex_to_rgba(hex_color, alpha=0.3):
        """Convert hex color to rgba string."""
        h = hex_color.lstrip('#')
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f'rgba({r},{g},{b},{alpha})'
    
    link_colors = [hex_to_rgba(STYLE['cluster_colors'][s % len(STYLE['cluster_colors'])], 0.3)
                   for s in sources]
    
    fig_sankey = go.Figure(go.Sankey(
        node=dict(
            label=all_labels,
            color=node_colors,
            pad=15, thickness=20,
        ),
        link=dict(
            source=sources, target=targets, value=values,
            color=link_colors,
        ),
    ))
    fig_sankey.update_layout(
        **base_layout(height=550, width=1000,
                      margin=dict(l=20, r=20, t=30, b=20)),
    )
    save_html(fig_sankey, f'ext3_od_sankey_{year_label}.html')

# -- 3b. OD flow change heatmap (cluster x cluster) --
# Pivot each OD matrix to N_CLUSTERS x N_CLUSTERS
def pivot_od(od_df, n_clusters):
    matrix = np.zeros((n_clusters, n_clusters))
    total = od_df['trips'].sum()
    for _, row in od_df.iterrows():
        pu, do = int(row['PU_cluster']), int(row['DO_cluster'])
        if pu < n_clusters and do < n_clusters:
            matrix[pu, do] = 100 * row['trips'] / total
    return matrix

od_matrix_2018 = pivot_od(od_2018, N_CLUSTERS)
od_matrix_2025 = pivot_od(od_2025, N_CLUSTERS)

# Remap 2025 clusters to match 2018 ordering
od_matrix_2025_aligned = np.zeros_like(od_matrix_2025)
for i18, i25 in cluster_matches.items():
    for j18, j25 in cluster_matches.items():
        od_matrix_2025_aligned[i18, j18] = od_matrix_2025[i25, j25]

od_diff = od_matrix_2025_aligned - od_matrix_2018
cluster_short = [short_label(cnames_2018[i]) for i in range(N_CLUSTERS)]

fig_od_heat = make_subplots(
    rows=1, cols=3,
    subplot_titles=('2018 OD Share (%)', '2025 OD Share (%)', 'Change (pp)'),
    horizontal_spacing=0.08,
)

for col_idx, (matrix, cscale, show_cb) in enumerate([
    (od_matrix_2018, 'Viridis', False),
    (od_matrix_2025_aligned, 'Viridis', True),
    (od_diff, 'RdBu', True),
], 1):
    fig_od_heat.add_trace(go.Heatmap(
        z=matrix, x=cluster_short, y=cluster_short,
        colorscale=cscale,
        zmid=0 if col_idx == 3 else None,
        showscale=show_cb,
        colorbar=dict(
            x=0.65 if col_idx == 2 else 0.99,
            ticksuffix='%' if col_idx < 3 else ' pp',
            title=dict(font=dict(family=STYLE['font_family'])),
            tickfont=dict(family=STYLE['font_family']),
        ) if show_cb else None,
        hovertemplate='PU: %{y}<br>DO: %{x}<br>Value: %{z:.2f}<extra></extra>',
    ), row=1, col=col_idx)

fig_od_heat.update_layout(**base_layout(height=500, width=1200))
fig_od_heat.update_xaxes(title_text='Dropoff Cluster')
fig_od_heat.update_yaxes(title_text='Pickup Cluster', col=1)

save_html(fig_od_heat, 'ext3_od_flow_heatmaps.html')

# -- 3c. Dropoff density change map (mirrors pickup density change) --
print("[3c] Dropoff density change map...")

do_counts_2018 = df_2018.dropna(subset=['DO_zone_id']).groupby(
    ['DO_zone_id', 'DO_zone_name', 'DO_borough']
).size().reset_index(name='count_2018')
do_counts_2025 = df_2025.dropna(subset=['DO_zone_id']).groupby(
    ['DO_zone_id', 'DO_zone_name', 'DO_borough']
).size().reset_index(name='count_2025')

do_counts_2018.columns = ['zone_id', 'zone_name', 'borough', 'count_2018']
do_counts_2025.columns = ['zone_id', 'zone_name', 'borough', 'count_2025']

do_comparison = do_counts_2018.merge(
    do_counts_2025, on=['zone_id', 'zone_name', 'borough'], how='outer'
).fillna(0)
do_comparison['share_2018'] = 100 * do_comparison['count_2018'] / do_comparison['count_2018'].sum()
do_comparison['share_2025'] = 100 * do_comparison['count_2025'] / do_comparison['count_2025'].sum()
do_comparison['share_change'] = do_comparison['share_2025'] - do_comparison['share_2018']

do_change = do_comparison[['zone_id', 'zone_name', 'borough', 'share_change']].copy()
do_change['zone_id'] = do_change['zone_id'].astype(float).astype(int).astype(str)

zone_ids_do = set(do_change['zone_id'].values)
filtered_geo_do = {
    'type': 'FeatureCollection',
    'features': [f for f in taxi_zones_geo_4326['features']
                 if f['properties']['LocationID'] in zone_ids_do]
}

cap_do = do_change['share_change'].abs().quantile(0.95)
do_change['share_change_capped'] = do_change['share_change'].clip(-cap_do, cap_do)

fig_do_change = go.Figure()
fig_do_change.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[0] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))
fig_do_change.add_trace(go.Choroplethmap(
    geojson=filtered_geo_do,
    locations=do_change['zone_id'],
    featureidkey='properties.LocationID',
    z=do_change['share_change_capped'],
    colorscale='RdBu', zmid=0,
    marker_opacity=0.85,
    marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(
        title=dict(text='Change (pp)', font=dict(family=STYLE['font_family'])),
        ticksuffix=' pp', x=0.99,
        tickfont=dict(family=STYLE['font_family']),
    ),
    customdata=np.column_stack([
        do_change['zone_name'], do_change['borough'], do_change['share_change']
    ]),
    hovertemplate=(
        '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        'DO Change: %{customdata[2]:.2f} pp<extra></extra>'
    ),
))
fig_do_change.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'],
    map_zoom=STYLE['map_zoom'],
    map_center=dict(**STYLE['map_center']),
)
save_html(fig_do_change, 'ext3_dropoff_density_change.html')


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 4: TRIP DISTANCE ANALYSIS
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[4] TRIP DISTANCE ANALYSIS (HAVERSINE PU-DO)")
print("=" * 70)

def haversine_km_vectorized(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance (numpy arrays, returns km)."""
    lat1, lon1, lat2, lon2 = (np.radians(x) for x in [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


def compute_trip_distances(df, label=''):
    """Compute great-circle distance between PU and DO zone centroids (vectorized)."""
    valid = df.dropna(subset=['PU_lat', 'PU_lon', 'DO_lat', 'DO_lon']).copy()
    valid['trip_dist_km'] = haversine_km_vectorized(
        valid['PU_lat'].values, valid['PU_lon'].values,
        valid['DO_lat'].values, valid['DO_lon'].values,
    )
    # Filter out zero-distance (same zone) and extreme outliers
    valid = valid[(valid['trip_dist_km'] > 0) & (valid['trip_dist_km'] < 100)]
    print(f"  [{label}] Computed distances for {len(valid):,} trips")
    print(f"    Mean: {valid['trip_dist_km'].mean():.2f} km, "
          f"Median: {valid['trip_dist_km'].median():.2f} km")
    return valid

# Vectorized haversine is fast, no need for subsampling
print("[4.1] Computing 2018 distances...")
df_2018_dist = compute_trip_distances(df_2018, '2018')
print("[4.2] Computing 2025 distances...")
df_2025_dist = compute_trip_distances(df_2025, '2025')

# -- 4a. Overall distance distribution --
fig_dist = go.Figure()
for label, df_d, color in [('2018', df_2018_dist, STYLE['year_2018']),
                            ('2025', df_2025_dist, STYLE['year_2025'])]:
    fig_dist.add_trace(go.Histogram(
        x=df_d['trip_dist_km'],
        histnorm='probability density',
        nbinsx=80, name=label,
        marker_color=color, opacity=0.65,
        hovertemplate='%{x:.1f} km: density=%{y:.4f}<extra>' + label + '</extra>',
    ))

fig_dist.update_layout(
    **base_layout(height=500, width=900),
    xaxis=styled_axis(title_text='Trip Distance (km)', range=[0, 40]),
    yaxis=styled_axis(title_text='Density'),
    barmode='overlay',
    legend=dict(x=0.75, y=0.95, font=dict(family=STYLE['font_family'])),
)
save_html(fig_dist, 'ext4_trip_distance_distribution.html')

# -- 4b. Distance by cluster --
dist_by_cluster = []
for i in range(N_CLUSTERS):
    d18 = df_2018_dist[df_2018_dist['PU_cluster'] == i]['trip_dist_km']
    i25 = cluster_matches[i]
    d25 = df_2025_dist[df_2025_dist['PU_cluster'] == i25]['trip_dist_km']
    dist_by_cluster.append({
        'cluster': short_label(cnames_2018[i]),
        'mean_2018': d18.mean() if len(d18) > 0 else np.nan,
        'mean_2025': d25.mean() if len(d25) > 0 else np.nan,
        'median_2018': d18.median() if len(d18) > 0 else np.nan,
        'median_2025': d25.median() if len(d25) > 0 else np.nan,
    })

dist_cluster_df = pd.DataFrame(dist_by_cluster)

fig_dist_cluster = go.Figure()
fig_dist_cluster.add_trace(go.Bar(
    x=dist_cluster_df['cluster'], y=dist_cluster_df['median_2018'],
    name='2018 (median)', marker_color=STYLE['year_2018'],
    hovertemplate='%{x}<br>Median: %{y:.2f} km<extra>2018</extra>',
))
fig_dist_cluster.add_trace(go.Bar(
    x=dist_cluster_df['cluster'], y=dist_cluster_df['median_2025'],
    name='2025 (median)', marker_color=STYLE['year_2025'],
    hovertemplate='%{x}<br>Median: %{y:.2f} km<extra>2025</extra>',
))

fig_dist_cluster.update_layout(
    **base_layout(height=500, width=900),
    xaxis=styled_axis(title_text='Pickup Cluster'),
    yaxis=styled_axis(title_text='Median Trip Distance (km)'),
    barmode='group',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
)
save_html(fig_dist_cluster, 'ext4_distance_by_cluster.html')

# Summary stats
print("\n  Distance summary:")
print(f"    2018: mean={df_2018_dist['trip_dist_km'].mean():.2f} km, "
      f"median={df_2018_dist['trip_dist_km'].median():.2f} km")
print(f"    2025: mean={df_2025_dist['trip_dist_km'].mean():.2f} km, "
      f"median={df_2025_dist['trip_dist_km'].median():.2f} km")


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 5: DEMAND-FEATURE CLUSTERING
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[5] DEMAND-FEATURE CLUSTERING (HOURLY PROFILES)")
print("=" * 70)

def build_zone_hourly_profiles(df, zone_col='PU_zone_id'):
    """Build a 24-dimensional hourly demand profile per zone."""
    hourly = df.groupby([zone_col, 'hour']).size().reset_index(name='trips')
    pivot = hourly.pivot(index=zone_col, columns='hour', values='trips').fillna(0)
    # Normalize to shares within each zone
    row_sums = pivot.sum(axis=1)
    pivot_norm = pivot.div(row_sums, axis=0)
    # Fill any fully-zero zones
    pivot_norm = pivot_norm.fillna(0)
    return pivot_norm


def cluster_demand_profiles(pivot_norm, n_clusters=4):
    """K-means on standardized hourly profiles."""
    scaler = StandardScaler()
    X = scaler.fit_transform(pivot_norm.values)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    return labels, km, scaler


N_DEMAND_CLUSTERS = 4  # fewer than geographic to keep it interpretable

for year_label, df_year, c_names_year in [('2018', df_2018, cnames_2018),
                                            ('2025', df_2025, cnames_2025)]:
    print(f"\n[5.{year_label}] Building demand profiles...")
    
    profiles = build_zone_hourly_profiles(df_year)
    # Filter zones with at least 50 trips
    zone_totals = df_year.groupby('PU_zone_id').size()
    valid_zones = zone_totals[zone_totals >= 50].index
    profiles = profiles.loc[profiles.index.isin(valid_zones)]
    
    labels, km_demand, _ = cluster_demand_profiles(profiles, N_DEMAND_CLUSTERS)
    profiles['demand_cluster'] = labels
    
    # Name clusters by their peak hour pattern
    demand_cluster_names = {}
    for c in range(N_DEMAND_CLUSTERS):
        mask = profiles['demand_cluster'] == c
        avg_profile = profiles.loc[mask].drop(columns='demand_cluster').mean()
        peak_hour = avg_profile.idxmax()
        n_zones = mask.sum()
        if peak_hour <= 9:
            pattern = 'Morning-peak'
        elif peak_hour <= 14:
            pattern = 'Midday-peak'
        elif peak_hour <= 19:
            pattern = 'Evening-peak'
        else:
            pattern = 'Night-peak'
        demand_cluster_names[c] = f"{pattern} ({n_zones} zones)"
    
    profiles['demand_cluster_name'] = profiles['demand_cluster'].map(demand_cluster_names)
    
    print(f"  Demand clusters ({year_label}):")
    for c in range(N_DEMAND_CLUSTERS):
        print(f"    {demand_cluster_names[c]}")
    
    # -- Plot: average hourly profile per demand cluster --
    demand_colors = ['#264653', '#2a9d8f', '#e9c46a', '#e76f51']
    
    fig_demand = go.Figure()
    for c in range(N_DEMAND_CLUSTERS):
        mask = profiles['demand_cluster'] == c
        avg = profiles.loc[mask].drop(columns=['demand_cluster', 'demand_cluster_name']).mean()
        fig_demand.add_trace(go.Scatter(
            x=list(range(24)), y=avg.values,
            mode='lines+markers', name=demand_cluster_names[c],
            line=dict(color=demand_colors[c % len(demand_colors)], width=2.5),
            marker=dict(size=5),
            hovertemplate='Hour %{x}: %{y:.3f}<extra>' + demand_cluster_names[c] + '</extra>',
        ))
    
    fig_demand.update_layout(
        **base_layout(height=450, width=900),
        xaxis=styled_axis(title_text='Hour of Day'),
        yaxis=styled_axis(title_text='Share of Zone Trips'),
        legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
    )
    save_html(fig_demand, f'ext5_demand_profiles_{year_label}.html')
    
    # -- Map: demand cluster assignment --
    zone_demand = profiles[['demand_cluster', 'demand_cluster_name']].reset_index()
    zone_demand.columns = ['zone_id', 'demand_cluster', 'demand_cluster_name']
    zone_demand['zone_id'] = zone_demand['zone_id'].astype(float).astype(int).astype(str)
    
    # Merge zone names
    zc_lookup = zone_centroids.drop_duplicates(subset='zone_id').set_index('zone_id')
    zone_demand['zone_name'] = zone_demand['zone_id'].astype(int).map(zc_lookup['zone_name'])
    zone_demand['borough'] = zone_demand['zone_id'].astype(int).map(zc_lookup['borough'])
    
    zone_ids_demand = set(zone_demand['zone_id'].values)
    filtered_geo_demand = {
        'type': 'FeatureCollection',
        'features': [f for f in taxi_zones_geo_4326['features']
                     if f['properties']['LocationID'] in zone_ids_demand]
    }
    
    demand_color_map = {
        demand_cluster_names[c]: demand_colors[c % len(demand_colors)]
        for c in range(N_DEMAND_CLUSTERS)
    }
    
    fig_dmap = go.Figure()
    fig_dmap.add_trace(go.Choroplethmap(
        geojson=taxi_zones_geo_4326,
        locations=all_zone_ids,
        featureidkey='properties.LocationID',
        z=[1] * len(all_zone_ids),
        colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
        marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
        showscale=False, hoverinfo='skip',
    ))
    
    fig_tmp = px.choropleth_map(
        zone_demand, geojson=filtered_geo_demand,
        locations='zone_id', featureidkey='properties.LocationID',
        color='demand_cluster_name', color_discrete_map=demand_color_map,
        map_style=STYLE['map_style'], zoom=STYLE['map_zoom'],
        center=STYLE['map_center'], opacity=0.85,
        hover_data={'zone_id': False, 'zone_name': True, 'borough': True,
                    'demand_cluster_name': True},
        labels={'demand_cluster_name': 'Demand Type'},
    )
    fig_tmp.update_traces(marker=dict(line=dict(width=0.6, color='rgba(255,255,255,0.7)')))
    for trace in fig_tmp.data:
        fig_dmap.add_trace(trace)
    
    fig_dmap.update_layout(
        **base_layout(height=650, width=1100, margin=STYLE['margin_map']),
        legend=dict(
            title="Demand Type", yanchor="top", y=0.99, xanchor="left", x=0.01,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#dde1e7", borderwidth=1,
            font=dict(size=STYLE['legend_size'], family=STYLE['font_family']),
        ),
        map=dict(style=STYLE['map_style'], center=STYLE['map_center'],
                 zoom=STYLE['map_zoom']),
    )
    save_html(fig_dmap, f'ext5_demand_cluster_map_{year_label}.html')


# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART 1 COMPLETE")
print("=" * 70)
print(f"\nOutputs saved to: {OUTPUT_DIR}")
print("  Extension charts:")
ext1_files = [
    'ext1_market_share_breakdown.html',
    'ext1_market_share_stacked.html',
    'ext2_timeseries_gini_moran.html',
    'ext2_uber_share_timeseries.html',
    'ext3_od_sankey_2018.html',
    'ext3_od_sankey_2025.html',
    'ext3_od_flow_heatmaps.html',
    'ext3_dropoff_density_change.html',
    'ext4_trip_distance_distribution.html',
    'ext4_distance_by_cluster.html',
    'ext5_demand_profiles_2018.html',
    'ext5_demand_profiles_2025.html',
    'ext5_demand_cluster_map_2018.html',
    'ext5_demand_cluster_map_2025.html',
]
for f in ext1_files:
    status = "OK" if os.path.exists(OUTPUT_DIR + f) else "SKIPPED"
    print(f"    [{status}] {f}")

print("\n--- Methods Summary ---")
print("  1. Market share: Uber vs Lyft vs Other FHV from dispatching_base_num / hvfhs_license_num")
print("  2. Time series: Gini + Moran's I computed per available January parquet (2018-2025)")
print("  3. OD flows: cluster-to-cluster Sankey + heatmap + dropoff density change choropleth")
print("  4. Trip distance: Haversine between PU/DO zone centroids, vectorized numpy (full dataset)")
print("  5. Demand clustering: K-means on 24-dim hourly profile (standardized), k=4")

LOADING COMMON DATA
Loaded 263 zone centroids

LOADING MAIN DATASETS

[Loading 2018] /Users/leoss/Desktop/Portfolio/Website-/projects/uber/data/fhv_tripdata_2018-01.parquet
  Total rows: 19,808,094
  Uber trips: 4,502,999 (22.7%)

[Loading 2025] /Users/leoss/Desktop/Portfolio/Website-/projects/uber/data/fhvhv_tripdata_2025-01.parquet
  Total rows: 20,405,666
  Uber trips: 15,356,455 (75.3%)


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_1118/2978125917.py:265: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[211 181  76 ... 235 235 133]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.




[1] CONFOUNDER ANALYSIS: MARKET SHARE BREAKDOWN
[1.1] 2018 market share breakdown...
[1.2] 2025 market share breakdown...

  Market share breakdown:
    2018 Uber:    4,502,999 (22.7%)
    2018 Lyft:    3,141,012 (15.9%)
    2018 Other FHV:   12,164,083 (61.4%)
    2025 Uber:   15,356,455 (75.3%)
    2025 Lyft:    5,049,211 (24.7%)
    2025 Other FHV:            0 (0.0%)
  Saved: ext1_market_share_breakdown.html
  Saved: ext1_market_share_stacked.html

[2] MULTI-YEAR TIME SERIES: GINI AND MORAN'S I
  [Processing 2018]...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_1118/2978125917.py:161: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



    Uber share: 22.7%, Gini: 0.516, Moran's I: 0.544 (p=0.0010)
  [SKIP] 2019: file not found (fhv_tripdata_2019-01.parquet)
  [SKIP] 2020: file not found (fhvhv_tripdata_2020-01.parquet)
  [SKIP] 2021: file not found (fhvhv_tripdata_2021-01.parquet)
  [SKIP] 2022: file not found (fhvhv_tripdata_2022-01.parquet)
  [SKIP] 2023: file not found (fhvhv_tripdata_2023-01.parquet)
  [SKIP] 2024: file not found (fhvhv_tripdata_2024-01.parquet)
  [Processing 2025]...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_1118/2978125917.py:161: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



    Uber share: 75.3%, Gini: 0.445, Moran's I: 0.315 (p=0.0010)
  Saved: ext2_timeseries_gini_moran.html
  Saved: ext2_uber_share_timeseries.html

[3] DROPOFF AND OD FLOW ANALYSIS
  [2018] OD matrix: 4,364,044 valid trips, 36 cluster pairs
  [2025] OD matrix: 14,804,246 valid trips, 36 cluster pairs
  Saved: ext3_od_sankey_2018.html
  Saved: ext3_od_sankey_2025.html
  Saved: ext3_od_flow_heatmaps.html
[3c] Dropoff density change map...
  Saved: ext3_dropoff_density_change.html

[4] TRIP DISTANCE ANALYSIS (HAVERSINE PU-DO)
[4.1] Computing 2018 distances...
  [2018] Computed distances for 4,116,783 trips
    Mean: 4.96 km, Median: 3.46 km
[4.2] Computing 2025 distances...
  [2025] Computed distances for 13,599,708 trips
    Mean: 5.27 km, Median: 3.56 km
  Saved: ext4_trip_distance_distribution.html
  Saved: ext4_distance_by_cluster.html

  Distance summary:
    2018: mean=4.96 km, median=3.46 km
    2025: mean=5.27 km, median=3.56 km

[5] DEMAND-FEATURE CLUSTERING (HOURLY PROFILES)

[5.

In [ ]:
"""
Uber NYC Analysis - Extensions Part 2 (Suggestions 6-10)
========================================================
Standalone script that adds five more analyses to the original Uber NYC project.


6. Boundary-independent localization metric (zone-level, not cluster-dependent)
7. Compositional temporal analysis (within-cluster temporal stability test)
8. Graduated prediction scorecard (HTML output with strength-of-evidence ratings)
9. Bootstrap confidence intervals on Gini coefficient
10. Properly present mismatch statistics (KS + Levene with visualization)

Requirements:
    - Same data files as the original notebook (Jan 2018 and Jan 2025 parquets,
      zone_centroids.csv, taxi_zones shapefile)
    - Part 1 script does NOT need to have run first; this is fully independent.

Outputs saved to OUTPUT_DIR as interactive Plotly HTML files.

Methods:
    - Zone-level localization: for each PU zone, fraction of DO trips landing in same
      zone or K nearest zones (K=5, based on centroid distance). Computed on aggregated
      OD pair counts (thousands of pairs) rather than individual trips (millions of rows).
    - Compositional temporal: Jensen-Shannon divergence between 2018 and 2025 hourly
      profiles per cluster, compared to aggregate JSD.
    - Bootstrap Gini: 1000-iteration bootstrap resampling of zone-level trip counts
      (resampling zones, not individual trips), computing Gini each iteration for 95% CIs.
    - Mismatch statistics: PU/(PU+DO) ratio distributions plotted as histograms with
      KS and Levene test results annotated.
"""

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.stats import ks_2samp, levene, entropy
from math import radians, cos, sin, asin, sqrt
import geopandas as gpd
import libpysal
from esda.moran import Moran
import json
import gc
import os

# ==========================================================================
# CONFIGURATION (matches original notebook)
# ==========================================================================
BASE_DIR = '/Users/leoss/Desktop/Portfolio/Website-/'
DATA_DIR = BASE_DIR + 'projects/uber/data/'
OUTPUT_DIR = BASE_DIR + 'projects/uber/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_2018 = DATA_DIR + 'fhv_tripdata_2018-01.parquet'
PATH_2025 = DATA_DIR + 'fhvhv_tripdata_2025-01.parquet'
PATH_CENTROIDS = DATA_DIR + 'zone_centroids.csv'
PATH_SHAPEFILE = DATA_DIR + 'taxi_zones/taxi_zones.shp'

SAMPLE_SIZE = 20_000_000
N_CLUSTERS = 6

UBER_2018_BASES = ['B02512', 'B02598', 'B02617', 'B02682', 'B02764', 'B02765', 'B02835', 'B02836']
UBER_2025_LICENSE = 'HV0003'

AIRPORT_ZONE_IDS = {132, 138, 1}

STYLE = {
    'font_family': 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size': 11,
    'axis_title_size': 13,
    'legend_size': 11,
    'template': 'plotly_white',
    'plot_bg': 'rgba(0,0,0,0)',
    'paper_bg': 'white',
    'chart_height': 550,
    'margin_default': dict(l=60, r=40, t=20, b=50),
    'margin_map': dict(l=0, r=0, t=20, b=0),
    'grid_color': '#e5e7eb',
    'grid_width': 0.5,
    'hover_font_size': 13,
    'hover_font_color': '#1a2744',
    'year_2018': '#ff6b6b',
    'year_2025': '#4ecdc4',
    'cluster_colors': ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4'],
    'map_style': 'carto-positron-nolabels',
    'map_center': {'lat': 40.7128, 'lon': -73.9352},
    'map_zoom': 9.5,
}


def base_layout(height=None, width=None, **kwargs):
    layout = dict(
        title='',
        font=dict(family=STYLE['font_family']),
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        height=height or STYLE['chart_height'],
        margin=STYLE['margin_default'],
        hoverlabel=dict(
            font_size=STYLE['hover_font_size'],
            font_color=STYLE['hover_font_color'],
        ),
    )
    if width:
        layout['width'] = width
    layout.update(kwargs)
    return layout


def styled_axis(**kwargs):
    return dict(
        tickfont=dict(size=STYLE['tick_size']),
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'],
        gridwidth=STYLE['grid_width'],
        **kwargs,
    )


def save_html(fig, filename):
    fig.write_html(
        OUTPUT_DIR + filename,
        include_plotlyjs='cdn',
        config={'displayModeBar': False},
    )
    print(f"  Saved: {filename}")


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * 6371 * asin(sqrt(a))


def lorenz_data(zone_counts_series):
    vals = zone_counts_series.sort_values().values
    cum_zones = np.arange(1, len(vals) + 1) / len(vals)
    cum_trips = np.cumsum(vals) / vals.sum()
    return cum_zones, cum_trips


def gini_from_lorenz(cum_zones, cum_trips):
    return 1 - 2 * np.trapz(cum_trips, cum_zones)


def name_cluster_by_location(center_lat, center_lon, zone_centroids):
    zone_distances = np.sqrt(
        (zone_centroids['latitude'] - center_lat) ** 2 +
        (zone_centroids['longitude'] - center_lon) ** 2
    )
    nearest_idx = zone_distances.idxmin()
    nearest_zone = zone_centroids.iloc[nearest_idx]
    return f"{nearest_zone['borough']}: {nearest_zone['zone_name']}"


def match_clusters_by_proximity(centers_a, centers_b):
    distances = cdist(centers_a, centers_b, metric='cityblock')
    matches = {}
    used_b = set()
    pairs = sorted(
        [(i, j, distances[i, j]) for i in range(len(centers_a)) for j in range(len(centers_b))],
        key=lambda x: x[2]
    )
    for i, j, dist in pairs:
        if i not in matches and j not in used_b:
            matches[i] = j
            used_b.add(j)
    return matches


def merge_zone_info(df, zone_col, centroids, prefix):
    merged = df.merge(
        centroids[['zone_id', 'zone_name', 'borough', 'latitude', 'longitude']],
        left_on=zone_col, right_on='zone_id', how='left'
    )
    merged = merged.rename(columns={
        'zone_id': f'{prefix}_zone_id',
        'zone_name': f'{prefix}_zone_name',
        'borough': f'{prefix}_borough',
        'latitude': f'{prefix}_lat',
        'longitude': f'{prefix}_lon',
    })
    return merged


def mismatch_ratios(df, pu_col='PU_zone_id', do_col='DO_zone_id'):
    pu = df.groupby(pu_col).size().rename('pu')
    do = df.dropna(subset=[do_col]).groupby(do_col).size().rename('do')
    combined = pd.concat([pu, do], axis=1).fillna(0)
    combined['ratio'] = combined['pu'] / (combined['pu'] + combined['do'])
    combined['total'] = combined['pu'] + combined['do']
    return combined


def short_label(name):
    return name.split(':')[0].strip()


# ==========================================================================
# LOAD DATA
# ==========================================================================
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

zone_centroids = pd.read_csv(PATH_CENTROIDS)
gdf_raw = gpd.read_file(PATH_SHAPEFILE).to_crs(epsg=4326)
taxi_zones_geo_4326 = json.loads(gdf_raw.to_json())
for f in taxi_zones_geo_4326['features']:
    f['properties']['LocationID'] = str(int(f['properties']['LocationID']))
all_zone_ids = [f['properties']['LocationID'] for f in taxi_zones_geo_4326['features']]

# -- 2018 --
print("\n[Loading 2018]...")
table_2018 = pq.read_table(PATH_2018,
    columns=['pickup_datetime', 'PUlocationID', 'DOlocationID', 'dispatching_base_num'])
df_2018 = table_2018.to_pandas()
total_2018 = len(df_2018)
df_2018 = df_2018[df_2018['dispatching_base_num'].isin(UBER_2018_BASES)].copy()
uber_n_2018 = len(df_2018)
if len(df_2018) > SAMPLE_SIZE:
    df_2018 = df_2018.sample(n=SAMPLE_SIZE, random_state=42)

df_2018['pickup_datetime'] = pd.to_datetime(df_2018['pickup_datetime'])
df_2018['hour'] = df_2018['pickup_datetime'].dt.hour
df_2018['day_of_week'] = df_2018['pickup_datetime'].dt.dayofweek
df_2018['day_name'] = df_2018['pickup_datetime'].dt.day_name()

df_2018 = df_2018.dropna(subset=['PUlocationID'])
df_2018['PUlocationID'] = df_2018['PUlocationID'].astype(int)
df_2018 = merge_zone_info(df_2018, 'PUlocationID', zone_centroids, 'PU')
df_2018 = df_2018.dropna(subset=['PU_lat', 'PU_lon'])

df_2018['DOlocationID'] = pd.to_numeric(df_2018['DOlocationID'], errors='coerce')
df_2018.loc[df_2018['DOlocationID'].notna(), 'DOlocationID'] = \
    df_2018.loc[df_2018['DOlocationID'].notna(), 'DOlocationID'].astype(int)
df_2018 = merge_zone_info(df_2018, 'DOlocationID', zone_centroids, 'DO')

coords_2018 = df_2018[['PU_lat', 'PU_lon']].values
kmeans_2018 = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_2018['PU_cluster'] = kmeans_2018.fit_predict(coords_2018)
cnames_2018 = {i: name_cluster_by_location(*kmeans_2018.cluster_centers_[i], zone_centroids)
               for i in range(N_CLUSTERS)}
df_2018['PU_cluster_name'] = df_2018['PU_cluster'].map(cnames_2018)

do_mask = df_2018['DO_lat'].notna() & df_2018['DO_lon'].notna()
df_2018.loc[do_mask, 'DO_cluster'] = kmeans_2018.predict(
    df_2018.loc[do_mask, ['DO_lat', 'DO_lon']].values)

del table_2018
gc.collect()
print(f"  2018: {len(df_2018):,} Uber trips loaded")

# -- 2025 --
print("[Loading 2025]...")
table_2025 = pq.read_table(PATH_2025,
    columns=['pickup_datetime', 'PULocationID', 'DOLocationID', 'hvfhs_license_num'])
df_2025 = table_2025.to_pandas()
total_2025 = len(df_2025)
df_2025 = df_2025[df_2025['hvfhs_license_num'] == UBER_2025_LICENSE].copy()
uber_n_2025 = len(df_2025)
if len(df_2025) > SAMPLE_SIZE:
    df_2025 = df_2025.sample(n=SAMPLE_SIZE, random_state=42)

df_2025['pickup_datetime'] = pd.to_datetime(df_2025['pickup_datetime'])
df_2025['hour'] = df_2025['pickup_datetime'].dt.hour
df_2025['day_of_week'] = df_2025['pickup_datetime'].dt.dayofweek
df_2025['day_name'] = df_2025['pickup_datetime'].dt.day_name()

df_2025 = df_2025.dropna(subset=['PULocationID'])
df_2025['PULocationID'] = df_2025['PULocationID'].astype(int)
df_2025 = merge_zone_info(df_2025, 'PULocationID', zone_centroids, 'PU')
df_2025 = df_2025.dropna(subset=['PU_lat', 'PU_lon'])

df_2025['DOLocationID'] = pd.to_numeric(df_2025['DOLocationID'], errors='coerce')
df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'] = \
    df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'].astype(int)
df_2025 = merge_zone_info(df_2025, 'DOLocationID', zone_centroids, 'DO')

coords_2025 = df_2025[['PU_lat', 'PU_lon']].values
kmeans_2025 = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_2025['PU_cluster'] = kmeans_2025.fit_predict(coords_2025)
cnames_2025 = {i: name_cluster_by_location(*kmeans_2025.cluster_centers_[i], zone_centroids)
               for i in range(N_CLUSTERS)}
df_2025['PU_cluster_name'] = df_2025['PU_cluster'].map(cnames_2025)

do_mask = df_2025['DO_lat'].notna() & df_2025['DO_lon'].notna()
df_2025.loc[do_mask, 'DO_cluster'] = kmeans_2025.predict(
    df_2025.loc[do_mask, ['DO_lat', 'DO_lon']].values)

del table_2025
gc.collect()
print(f"  2025: {len(df_2025):,} Uber trips loaded")

centers_2018 = kmeans_2018.cluster_centers_
centers_2025 = kmeans_2025.cluster_centers_
cluster_matches = match_clusters_by_proximity(centers_2018, centers_2025)


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 6: BOUNDARY-INDEPENDENT LOCALIZATION METRIC
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[6] BOUNDARY-INDEPENDENT LOCALIZATION (K-NEAREST ZONE)")
print("=" * 70)

# Precompute K nearest zones for each zone (based on centroid distances)
K_NEAREST = 5

zc = zone_centroids.drop_duplicates(subset='zone_id').copy()
zc = zc.set_index('zone_id')
zone_ids_all = zc.index.values

# Distance matrix between all zone centroids
n_zones = len(zone_ids_all)
zone_coords = zc[['latitude', 'longitude']].values
zone_dist_matrix = cdist(zone_coords, zone_coords, metric='euclidean')

# For each zone, find K nearest (excluding self)
nearest_zones = {}
for i, zid in enumerate(zone_ids_all):
    dists = zone_dist_matrix[i]
    sorted_idx = np.argsort(dists)
    # sorted_idx[0] is self (distance=0), take next K
    nearest = [zone_ids_all[j] for j in sorted_idx[1:K_NEAREST + 1]]
    nearest_zones[zid] = set(nearest) | {zid}  # include self


def compute_zone_localization(df, nearest_zones_dict, pu_zone_col='PU_zone_id',
                               do_zone_col='DO_zone_id'):
    """For each PU zone, fraction of trips where DO is in same zone or K nearest.
    Uses vectorized set-lookup via a precomputed mapping instead of row-by-row apply."""
    valid = df.dropna(subset=[pu_zone_col, do_zone_col]).copy()
    valid[pu_zone_col] = valid[pu_zone_col].astype(int)
    valid[do_zone_col] = valid[do_zone_col].astype(int)
    
    # Build a frozenset lookup for fast membership testing
    # For each (PU_zone, DO_zone) pair, check membership at the group level
    # instead of row-by-row
    od_counts = valid.groupby([pu_zone_col, do_zone_col]).size().reset_index(name='trips')
    
    od_counts['is_local'] = od_counts.apply(
        lambda r: int(r[do_zone_col]) in nearest_zones_dict.get(int(r[pu_zone_col]), set()),
        axis=1
    )
    
    # Now aggregate: this runs on ~unique zone pairs (thousands), not millions of trips
    total_by_zone = od_counts.groupby(pu_zone_col)['trips'].sum().rename('total')
    local_by_zone = od_counts[od_counts['is_local']].groupby(pu_zone_col)['trips'].sum().rename('local')
    
    zone_local = pd.concat([total_by_zone, local_by_zone], axis=1).fillna(0)
    zone_local['local_share'] = 100 * zone_local['local'] / zone_local['total']
    return zone_local


print(f"[6.1] Computing localization (K={K_NEAREST} nearest zones)...")
local_2018 = compute_zone_localization(df_2018, nearest_zones)
local_2025 = compute_zone_localization(df_2025, nearest_zones)

mean_local_2018 = local_2018['local_share'].mean()
mean_local_2025 = local_2025['local_share'].mean()
median_local_2018 = local_2018['local_share'].median()
median_local_2025 = local_2025['local_share'].median()

print(f"  2018: mean local share = {mean_local_2018:.1f}%, median = {median_local_2018:.1f}%")
print(f"  2025: mean local share = {mean_local_2025:.1f}%, median = {median_local_2025:.1f}%")

# -- Distribution plot --
fig_local = go.Figure()
fig_local.add_trace(go.Histogram(
    x=local_2018['local_share'], nbinsx=40, name='2018',
    marker_color=STYLE['year_2018'], opacity=0.6,
    hovertemplate='%{x:.1f}%: count=%{y}<extra>2018</extra>',
))
fig_local.add_trace(go.Histogram(
    x=local_2025['local_share'], nbinsx=40, name='2025',
    marker_color=STYLE['year_2025'], opacity=0.6,
    hovertemplate='%{x:.1f}%: count=%{y}<extra>2025</extra>',
))

# Add mean lines
fig_local.add_vline(x=mean_local_2018, line_dash='dash',
                     line_color=STYLE['year_2018'], line_width=2,
                     annotation_text=f'2018 mean: {mean_local_2018:.1f}%',
                     annotation_position='top left')
fig_local.add_vline(x=mean_local_2025, line_dash='dash',
                     line_color=STYLE['year_2025'], line_width=2,
                     annotation_text=f'2025 mean: {mean_local_2025:.1f}%',
                     annotation_position='top right')

fig_local.update_layout(
    **base_layout(height=500, width=900),
    xaxis=styled_axis(title_text=f'% of Trips to Same or {K_NEAREST}-Nearest Zones'),
    yaxis=styled_axis(title_text='Number of Zones'),
    barmode='overlay',
    legend=dict(x=0.75, y=0.95, font=dict(family=STYLE['font_family'])),
)
save_html(fig_local, 'ext6_zone_localization_distribution.html')

# -- Map: localization change per zone --
local_merged = local_2018[['local_share']].rename(columns={'local_share': 'local_2018'}).join(
    local_2025[['local_share']].rename(columns={'local_share': 'local_2025'}),
    how='outer'
).fillna(0)
local_merged['change'] = local_merged['local_2025'] - local_merged['local_2018']
local_merged = local_merged.reset_index()
local_merged.columns = ['zone_id', 'local_2018', 'local_2025', 'change']
local_merged['zone_id'] = local_merged['zone_id'].astype(str)

zone_ids_local = set(local_merged['zone_id'].values)
filtered_geo_local = {
    'type': 'FeatureCollection',
    'features': [f for f in taxi_zones_geo_4326['features']
                 if f['properties']['LocationID'] in zone_ids_local]
}

cap_local = local_merged['change'].abs().quantile(0.95)
local_merged['change_capped'] = local_merged['change'].clip(-cap_local, cap_local)

zc_lookup = zone_centroids.drop_duplicates(subset='zone_id').set_index('zone_id')
local_merged['zone_name'] = local_merged['zone_id'].astype(int).map(zc_lookup['zone_name'])
local_merged['borough'] = local_merged['zone_id'].astype(int).map(zc_lookup['borough'])

fig_local_map = go.Figure()
fig_local_map.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[0] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))
fig_local_map.add_trace(go.Choroplethmap(
    geojson=filtered_geo_local,
    locations=local_merged['zone_id'],
    featureidkey='properties.LocationID',
    z=local_merged['change_capped'],
    colorscale='RdBu', zmid=0,
    marker_opacity=0.85,
    marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(
        title=dict(text='Change (pp)', font=dict(family=STYLE['font_family'])),
        ticksuffix=' pp', x=0.99,
        tickfont=dict(family=STYLE['font_family']),
    ),
    customdata=np.column_stack([
        local_merged['zone_name'].fillna('Unknown'),
        local_merged['borough'].fillna('Unknown'),
        local_merged['change'],
        local_merged['local_2018'],
        local_merged['local_2025'],
    ]),
    hovertemplate=(
        '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        'Local share 2018: %{customdata[3]:.1f}%<br>'
        'Local share 2025: %{customdata[4]:.1f}%<br>'
        'Change: %{customdata[2]:.1f} pp<extra></extra>'
    ),
))
fig_local_map.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'],
    map_zoom=STYLE['map_zoom'],
    map_center=dict(**STYLE['map_center']),
)
save_html(fig_local_map, 'ext6_localization_change_map.html')


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 7: COMPOSITIONAL TEMPORAL ANALYSIS
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[7] COMPOSITIONAL TEMPORAL ANALYSIS (JENSEN-SHANNON DIVERGENCE)")
print("=" * 70)

def hourly_profile(df, filter_col=None, filter_val=None):
    """Return normalized 24-bin hourly distribution."""
    if filter_col is not None:
        df = df[df[filter_col] == filter_val]
    counts = df.groupby('hour').size().reindex(range(24), fill_value=0)
    total = counts.sum()
    if total == 0:
        return np.ones(24) / 24
    return (counts / total).values


def jsd(p, q):
    """Jensen-Shannon divergence (symmetric, bounded [0, 1] with log2)."""
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    # Add small epsilon to avoid log(0)
    eps = 1e-12
    p = p + eps
    q = q + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    return 0.5 * (entropy(p, m, base=2) + entropy(q, m, base=2))


# Aggregate JSD
agg_profile_2018 = hourly_profile(df_2018)
agg_profile_2025 = hourly_profile(df_2025)
agg_jsd = jsd(agg_profile_2018, agg_profile_2025)
print(f"  Aggregate hourly JSD: {agg_jsd:.6f}")

# Per-cluster JSD
cluster_jsds = []
for i in range(N_CLUSTERS):
    i25 = cluster_matches[i]
    p18 = hourly_profile(df_2018, 'PU_cluster', i)
    p25 = hourly_profile(df_2025, 'PU_cluster', i25)
    d = jsd(p18, p25)
    cluster_jsds.append({
        'cluster': i,
        'name': short_label(cnames_2018[i]),
        'jsd': d,
    })
    print(f"  Cluster {i} ({short_label(cnames_2018[i])}): JSD = {d:.6f}")

cjsd_df = pd.DataFrame(cluster_jsds)

# Bar chart: JSD per cluster + aggregate reference line
fig_jsd = go.Figure()
fig_jsd.add_trace(go.Bar(
    x=cjsd_df['name'], y=cjsd_df['jsd'],
    marker_color=[STYLE['cluster_colors'][i % len(STYLE['cluster_colors'])]
                  for i in range(N_CLUSTERS)],
    hovertemplate='%{x}<br>JSD: %{y:.6f}<extra></extra>',
))
fig_jsd.add_hline(
    y=agg_jsd, line_dash='dash', line_color='#333', line_width=2,
    annotation_text=f'Aggregate JSD: {agg_jsd:.4f}',
    annotation_position='top right',
)

fig_jsd.update_layout(
    **base_layout(height=450, width=850),
    xaxis=styled_axis(title_text='Cluster (matched 2018-2025)'),
    yaxis=styled_axis(title_text='Jensen-Shannon Divergence (hourly profile)'),
    showlegend=False,
)
save_html(fig_jsd, 'ext7_temporal_jsd_by_cluster.html')

# -- Compositional shift: how much of each hour's demand comes from each cluster --
print("[7.2] Hourly compositional breakdown...")

fig_comp = make_subplots(
    rows=1, cols=2,
    subplot_titles=('2018: Cluster Share by Hour', '2025: Cluster Share by Hour'),
    horizontal_spacing=0.08,
)

for col_idx, (df_year, cnames, year_label) in enumerate([
    (df_2018, cnames_2018, '2018'), (df_2025, cnames_2025, '2025')
], 1):
    hourly_by_cluster = df_year.groupby(['hour', 'PU_cluster']).size().reset_index(name='trips')
    hourly_total = df_year.groupby('hour').size().reset_index(name='total')
    hourly_by_cluster = hourly_by_cluster.merge(hourly_total, on='hour')
    hourly_by_cluster['share'] = 100 * hourly_by_cluster['trips'] / hourly_by_cluster['total']
    
    for c in range(N_CLUSTERS):
        subset = hourly_by_cluster[hourly_by_cluster['PU_cluster'] == c]
        fig_comp.add_trace(go.Bar(
            x=subset['hour'], y=subset['share'],
            name=short_label(cnames[c]),
            marker_color=STYLE['cluster_colors'][c % len(STYLE['cluster_colors'])],
            legendgroup=str(c),
            showlegend=(col_idx == 1),
            hovertemplate=f'{short_label(cnames[c])}<br>Hour %{{x}}: %{{y:.1f}}%<extra></extra>',
        ), row=1, col=col_idx)

fig_comp.update_layout(
    **base_layout(height=500, width=1200),
    barmode='stack',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])),
)
fig_comp.update_xaxes(title_text='Hour of Day')
fig_comp.update_yaxes(title_text='Cluster Share (%)', col=1)

save_html(fig_comp, 'ext7_hourly_composition_by_cluster.html')


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 8: GRADUATED PREDICTION SCORECARD (replaces all-green)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[8] GRADUATED PREDICTION SCORECARD")
print("=" * 70)

# Compute strength-of-evidence for each prediction
gini_2018 = gini_from_lorenz(*lorenz_data(df_2018.groupby('PU_zone_id').size()))
gini_2025 = gini_from_lorenz(*lorenz_data(df_2025.groupby('PU_zone_id').size()))

w = libpysal.weights.KNN.from_dataframe(gdf_raw, k=6)
w.transform = 'r'
gdf_base = gdf_raw.copy()
gdf_base['LocationID_int'] = gdf_base['LocationID'].astype(int)

def compute_morans(df_year):
    zone_counts = df_year.groupby('PU_zone_id').size()
    gdf_tmp = gdf_base.copy()
    gdf_tmp['trips'] = gdf_tmp['LocationID_int'].map(zone_counts).fillna(0)
    gdf_tmp['trip_share'] = 100 * gdf_tmp['trips'] / gdf_tmp['trips'].sum()
    mi = Moran(gdf_tmp['trip_share'].values, w)
    return mi.I, mi.p_sim

mi_2018, mi_p_2018 = compute_morans(df_2018)
mi_2025, mi_p_2025 = compute_morans(df_2025)

intra_2018 = df_2018.dropna(subset=['PU_cluster', 'DO_cluster'])
intra_2018_pct = 100 * (intra_2018['PU_cluster'] == intra_2018['DO_cluster']).mean()
intra_2025 = df_2025.dropna(subset=['PU_cluster', 'DO_cluster'])
intra_2025_pct = 100 * (intra_2025['PU_cluster'] == intra_2025['DO_cluster']).mean()

matched_shifts = [
    haversine_km(centers_2018[i18, 0], centers_2018[i18, 1],
                 centers_2025[i25, 0], centers_2025[i25, 1])
    for i18, i25 in cluster_matches.items()
]
avg_shift = np.mean(matched_shifts)

scorecard = [
    {
        'prediction': 'Increased airport demand',
        'result': 'JFK + LaGuardia combined pickup share rose 48%',
        'strength': 'Strong',
        'strength_color': '#2d6a4f',
        'notes': 'Clear directional change with large magnitude. However, airport ground '
                 'transportation policies also changed over this period (e.g. terminal access '
                 'restrictions for non-app hails), which could independently drive this.',
    },
    {
        'prediction': 'Manhattan core decline',
        'result': 'Top-3 zones lost 34-39% of their pickup share',
        'strength': 'Strong',
        'strength_color': '#2d6a4f',
        'notes': 'Large, consistent decline across multiple central zones. Post-pandemic '
                 'remote work is an equally plausible explanation that cannot be ruled out '
                 'with this design.',
    },
    {
        'prediction': 'Lower geographic concentration',
        'result': f'Gini: {gini_2018:.3f} to {gini_2025:.3f} (delta = {gini_2025 - gini_2018:.3f})',
        'strength': 'Moderate',
        'strength_color': '#b5830b',
        'notes': 'Direction is correct and magnitude is non-trivial. Without confidence '
                 'intervals or a counterfactual, it is unclear whether this exceeds what '
                 'market share growth alone would produce (more trips = more zones with any activity).',
    },
    {
        'prediction': 'Reduced spatial clustering',
        'result': f"Moran's I: {mi_2018:.3f} to {mi_2025:.3f}",
        'strength': 'Strong',
        'strength_color': '#2d6a4f',
        'notes': 'Nearly halved. Both values significant at p<0.001. This is the most '
                 'robust finding because Moran\'s I is less mechanically affected by volume '
                 'growth than the Gini coefficient.',
    },
    {
        'prediction': 'More localised matching',
        'result': f'Intra-cluster trips: {intra_2018_pct:.1f}% to {intra_2025_pct:.1f}%',
        'strength': 'Weak',
        'strength_color': '#c1121f',
        'notes': 'Clusters are fit independently per year, so boundary changes could produce '
                 'this result even with no change in trip geography. The zone-level localization '
                 f'metric (K={K_NEAREST} nearest zones) shows mean local share went from '
                 f'{mean_local_2018:.1f}% to {mean_local_2025:.1f}%, which is boundary-independent '
                 'but still modest.',
    },
    {
        'prediction': 'Stable geographic structure',
        'result': f'Mean centroid shift: {avg_shift:.2f} km; temporal profiles stable (JSD={agg_jsd:.4f})',
        'strength': 'Moderate',
        'strength_color': '#b5830b',
        'notes': 'Centroid shifts are small relative to NYC\'s extent (~30 km). But one cluster '
                 f'shifted 11.6 km (Brooklyn toward Staten Island), which is not trivially "stable". '
                 'Temporal JSD is low, supporting the argument, but compositional analysis shows '
                 'the spatial mix within each hour did shift.',
    },
]

# Generate HTML scorecard
scorecard_html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<style>
body { font-family: 'IBM Plex Sans', -apple-system, sans-serif; max-width: 1000px; margin: 2rem auto; padding: 0 1.5rem; color: #1a2744; }
h2 { margin-bottom: 1.5rem; }
table { width: 100%; border-collapse: collapse; font-size: 0.95rem; }
th { text-align: left; padding: 0.75rem 1rem; background: #f8f9fa; border-bottom: 2px solid #dde1e7; }
td { padding: 0.75rem 1rem; border-bottom: 1px solid #eee; vertical-align: top; }
.strength { font-weight: 600; padding: 0.2rem 0.6rem; border-radius: 3px; color: white; display: inline-block; font-size: 0.85rem; }
.notes { font-size: 0.88rem; color: #555; line-height: 1.5; }
</style>
</head>
<body>
<h2>Prediction Scorecard (Graduated)</h2>
<p style="color: #666; margin-bottom: 1.5rem;">
Each prediction is rated by strength of evidence: <span class="strength" style="background:#2d6a4f;">Strong</span> means the observed pattern is large, consistent, and less vulnerable to alternative explanations. <span class="strength" style="background:#b5830b;">Moderate</span> means directionally consistent but with meaningful confounders or methodological caveats. <span class="strength" style="background:#c1121f;">Weak</span> means the metric is fragile or the evidence is too sensitive to analytical choices.
</p>
<table>
<thead><tr><th>Prediction</th><th>Result</th><th>Strength</th><th>Notes</th></tr></thead>
<tbody>
"""

for s in scorecard:
    scorecard_html += f"""<tr>
<td><strong>{s['prediction']}</strong></td>
<td>{s['result']}</td>
<td><span class="strength" style="background:{s['strength_color']};">{s['strength']}</span></td>
<td class="notes">{s['notes']}</td>
</tr>
"""

scorecard_html += """</tbody></table>
</body></html>"""

with open(OUTPUT_DIR + 'ext8_graduated_scorecard.html', 'w') as f:
    f.write(scorecard_html)
print("  Saved: ext8_graduated_scorecard.html")


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 9: BOOTSTRAP CONFIDENCE INTERVALS ON GINI
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[9] BOOTSTRAP GINI CONFIDENCE INTERVALS")
print("=" * 70)

N_BOOTSTRAP = 1000

def bootstrap_gini(df, zone_col='PU_zone_id', n_boot=N_BOOTSTRAP):
    """Bootstrap Gini by resampling zone-level trip counts.
    
    Instead of resampling millions of individual trips (very slow), we resample
    zones with replacement and use their observed trip counts. This is a valid
    nonparametric bootstrap on the units of observation (zones) and runs in
    seconds instead of minutes.
    """
    zone_counts = df.groupby(zone_col).size().values
    n_zones = len(zone_counts)
    rng = np.random.RandomState(42)
    ginis = np.empty(n_boot)
    
    for b in range(n_boot):
        # Resample zones (with replacement), keep their trip counts
        sampled_counts = rng.choice(zone_counts, size=n_zones, replace=True)
        sorted_counts = np.sort(sampled_counts)
        cum_zones = np.arange(1, n_zones + 1) / n_zones
        cum_trips = np.cumsum(sorted_counts) / sorted_counts.sum()
        ginis[b] = 1 - 2 * np.trapz(cum_trips, cum_zones)
    
    return ginis

print("[9.1] Bootstrapping 2018 Gini...")
gini_boot_2018 = bootstrap_gini(df_2018)
print("[9.2] Bootstrapping 2025 Gini...")
gini_boot_2025 = bootstrap_gini(df_2025)

ci_2018 = np.percentile(gini_boot_2018, [2.5, 97.5])
ci_2025 = np.percentile(gini_boot_2025, [2.5, 97.5])

print(f"\n  2018 Gini: {gini_2018:.4f} [95% CI: {ci_2018[0]:.4f}, {ci_2018[1]:.4f}]")
print(f"  2025 Gini: {gini_2025:.4f} [95% CI: {ci_2025[0]:.4f}, {ci_2025[1]:.4f}]")
print(f"  CIs overlap: {ci_2018[0] < ci_2025[1] and ci_2025[0] < ci_2018[1]}")

# -- Bootstrap distribution plot --
fig_boot = go.Figure()
fig_boot.add_trace(go.Histogram(
    x=gini_boot_2018, nbinsx=50, name='2018',
    marker_color=STYLE['year_2018'], opacity=0.6,
    hovertemplate='Gini: %{x:.4f}<br>Count: %{y}<extra>2018</extra>',
))
fig_boot.add_trace(go.Histogram(
    x=gini_boot_2025, nbinsx=50, name='2025',
    marker_color=STYLE['year_2025'], opacity=0.6,
    hovertemplate='Gini: %{x:.4f}<br>Count: %{y}<extra>2025</extra>',
))

# CI lines
for ci, color, label in [
    (ci_2018, STYLE['year_2018'], '2018'),
    (ci_2025, STYLE['year_2025'], '2025'),
]:
    for bound, pos in [(ci[0], 'bottom left'), (ci[1], 'bottom right')]:
        fig_boot.add_vline(
            x=bound, line_dash='dot', line_color=color, line_width=1.5,
        )

fig_boot.update_layout(
    **base_layout(height=450, width=900),
    xaxis=styled_axis(title_text='Gini Coefficient'),
    yaxis=styled_axis(title_text='Bootstrap Count'),
    barmode='overlay',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
    annotations=[
        dict(
            x=0.99, y=0.99, xref='paper', yref='paper',
            xanchor='right', yanchor='top',
            text=(f"2018: {gini_2018:.4f} [{ci_2018[0]:.4f}, {ci_2018[1]:.4f}]<br>"
                  f"2025: {gini_2025:.4f} [{ci_2025[0]:.4f}, {ci_2025[1]:.4f}]"),
            showarrow=False,
            font=dict(size=11, family=STYLE['font_family']),
            bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1,
        )
    ],
)
save_html(fig_boot, 'ext9_bootstrap_gini.html')


# ══════════════════════════════════════════════════════════════════════════
# SUGGESTION 10: PROPERLY PRESENT MISMATCH STATISTICS
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("[10] PICKUP/DROPOFF MISMATCH STATISTICS")
print("=" * 70)

mr_2018 = mismatch_ratios(df_2018)
mr_2025 = mismatch_ratios(df_2025)

ks_stat, ks_p = ks_2samp(mr_2018['ratio'].dropna(), mr_2025['ratio'].dropna())
lev_stat, lev_p = levene(mr_2018['ratio'].dropna(), mr_2025['ratio'].dropna())

print(f"  KS test: stat={ks_stat:.4f}, p={ks_p:.4e}")
print(f"  Levene test: stat={lev_stat:.4f}, p={lev_p:.4e}")
print(f"  2018 ratio: mean={mr_2018['ratio'].mean():.3f}, std={mr_2018['ratio'].std():.3f}")
print(f"  2025 ratio: mean={mr_2025['ratio'].mean():.3f}, std={mr_2025['ratio'].std():.3f}")

# -- Distribution plot --
fig_mm = go.Figure()
fig_mm.add_trace(go.Histogram(
    x=mr_2018['ratio'].dropna(), nbinsx=50, name='2018',
    marker_color=STYLE['year_2018'], opacity=0.6,
    hovertemplate='Ratio: %{x:.3f}<br>Count: %{y}<extra>2018</extra>',
))
fig_mm.add_trace(go.Histogram(
    x=mr_2025['ratio'].dropna(), nbinsx=50, name='2025',
    marker_color=STYLE['year_2025'], opacity=0.6,
    hovertemplate='Ratio: %{x:.3f}<br>Count: %{y}<extra>2025</extra>',
))

# Perfect balance reference
fig_mm.add_vline(x=0.5, line_dash='dash', line_color='#333', line_width=1.5,
                  annotation_text='Perfect PU/DO balance',
                  annotation_position='top left')

fig_mm.update_layout(
    **base_layout(height=500, width=900),
    xaxis=styled_axis(title_text='PU / (PU + DO) Ratio'),
    yaxis=styled_axis(title_text='Number of Zones'),
    barmode='overlay',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
    annotations=[
        dict(
            x=0.99, y=0.99, xref='paper', yref='paper',
            xanchor='right', yanchor='top',
            text=(f"KS test: D={ks_stat:.4f}, p={ks_p:.2e}<br>"
                  f"Levene test: W={lev_stat:.4f}, p={lev_p:.2e}<br>"
                  f"<br>KS significant: distribution shape changed<br>"
                  f"Levene not significant: variance did not change"),
            showarrow=False,
            font=dict(size=10, family=STYLE['font_family']),
            bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1,
        )
    ],
)
save_html(fig_mm, 'ext10_mismatch_ratios.html')

# -- Mismatch map: change in PU/(PU+DO) ratio per zone --
mr_merged = mr_2018[['ratio']].rename(columns={'ratio': 'ratio_2018'}).join(
    mr_2025[['ratio']].rename(columns={'ratio': 'ratio_2025'}),
    how='outer',
)
mr_merged['ratio_change'] = mr_merged['ratio_2025'] - mr_merged['ratio_2018']
mr_merged = mr_merged.dropna(subset=['ratio_change']).reset_index()
mr_merged.columns = ['zone_id', 'ratio_2018', 'ratio_2025', 'ratio_change']
mr_merged['zone_id'] = mr_merged['zone_id'].astype(float).astype(int).astype(str)

mr_merged['zone_name'] = mr_merged['zone_id'].astype(int).map(zc_lookup['zone_name'])
mr_merged['borough'] = mr_merged['zone_id'].astype(int).map(zc_lookup['borough'])

zone_ids_mm = set(mr_merged['zone_id'].values)
filtered_geo_mm = {
    'type': 'FeatureCollection',
    'features': [f for f in taxi_zones_geo_4326['features']
                 if f['properties']['LocationID'] in zone_ids_mm]
}

cap_mm = mr_merged['ratio_change'].abs().quantile(0.95)
mr_merged['ratio_change_capped'] = mr_merged['ratio_change'].clip(-cap_mm, cap_mm)

fig_mm_map = go.Figure()
fig_mm_map.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326,
    locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[0] * len(all_zone_ids),
    colorscale=[[0, '#e8e8e8'], [1, '#e8e8e8']],
    marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip',
))
fig_mm_map.add_trace(go.Choroplethmap(
    geojson=filtered_geo_mm,
    locations=mr_merged['zone_id'],
    featureidkey='properties.LocationID',
    z=mr_merged['ratio_change_capped'],
    colorscale='RdBu', zmid=0,
    marker_opacity=0.85,
    marker_line_width=0.5,
    marker_line_color='rgba(255,255,255,0.6)',
    colorbar=dict(
        title=dict(text='Change in PU/(PU+DO)', font=dict(family=STYLE['font_family'])),
        x=0.99,
        tickfont=dict(family=STYLE['font_family']),
    ),
    customdata=np.column_stack([
        mr_merged['zone_name'].fillna('Unknown'),
        mr_merged['borough'].fillna('Unknown'),
        mr_merged['ratio_2018'],
        mr_merged['ratio_2025'],
        mr_merged['ratio_change'],
    ]),
    hovertemplate=(
        '<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
        '2018 ratio: %{customdata[2]:.3f}<br>'
        '2025 ratio: %{customdata[3]:.3f}<br>'
        'Change: %{customdata[4]:.3f}<extra></extra>'
    ),
))
fig_mm_map.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'],
    map_zoom=STYLE['map_zoom'],
    map_center=dict(**STYLE['map_center']),
)
save_html(fig_mm_map, 'ext10_mismatch_change_map.html')


# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("PART 2 COMPLETE")
print("=" * 70)
print(f"\nOutputs saved to: {OUTPUT_DIR}")
print("  Extension charts:")
ext2_files = [
    'ext6_zone_localization_distribution.html',
    'ext6_localization_change_map.html',
    'ext7_temporal_jsd_by_cluster.html',
    'ext7_hourly_composition_by_cluster.html',
    'ext8_graduated_scorecard.html',
    'ext9_bootstrap_gini.html',
    'ext10_mismatch_ratios.html',
    'ext10_mismatch_change_map.html',
]
for f in ext2_files:
    status = "OK" if os.path.exists(OUTPUT_DIR + f) else "SKIPPED"
    print(f"    [{status}] {f}")

print("\n--- Methods Summary ---")
print(f"  6. Zone localization: % of DO trips within same or K={K_NEAREST} nearest zones (Euclidean on centroids)")
print(f"  7. Compositional temporal: JSD between 2018/2025 hourly profiles, aggregate and per-cluster")
print(f"  8. Graduated scorecard: Strong/Moderate/Weak evidence ratings with methodological notes")
print(f"  9. Bootstrap Gini: {N_BOOTSTRAP} iterations resampling zones (not trips), 95% CI via percentile method")
print(f"  10. Mismatch: PU/(PU+DO) ratio distribution + KS/Levene tests + choropleth change map")

print("\n--- Key Findings ---")
print(f"  Gini 2018: {gini_2018:.4f} [{ci_2018[0]:.4f}, {ci_2018[1]:.4f}]")
print(f"  Gini 2025: {gini_2025:.4f} [{ci_2025[0]:.4f}, {ci_2025[1]:.4f}]")
print(f"  CIs do not overlap: {ci_2018[0] > ci_2025[1] or ci_2025[0] > ci_2018[1]}")
print(f"  Aggregate temporal JSD: {agg_jsd:.6f} (very low = stable temporal patterns)")
print(f"  Zone localization (mean): {mean_local_2018:.1f}% -> {mean_local_2025:.1f}%")
print(f"  KS mismatch: D={ks_stat:.4f}, p={ks_p:.2e}")
print(f"  Levene mismatch: W={lev_stat:.4f}, p={lev_p:.2e}")

LOADING DATA

[Loading 2018]...
  2018: 4,519,809 Uber trips loaded
[Loading 2025]...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/1721431353.py:282: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[211 181  76 ... 235 235 133]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'] = \


  2025: 15,447,196 Uber trips loaded

[6] BOUNDARY-INDEPENDENT LOCALIZATION (K-NEAREST ZONE)
[6.1] Computing localization (K=5 nearest zones)...
  2018: mean local share = 29.9%, median = 29.7%
  2025: mean local share = 33.4%, median = 32.7%
  Saved: ext6_zone_localization_distribution.html
  Saved: ext6_localization_change_map.html

[7] COMPOSITIONAL TEMPORAL ANALYSIS (JENSEN-SHANNON DIVERGENCE)
  Aggregate hourly JSD: 0.002548
  Cluster 0 (Manhattan): JSD = 0.002841
  Cluster 1 (Queens): JSD = 0.002777
  Cluster 2 (Bronx): JSD = 0.005707
  Cluster 3 (Brooklyn): JSD = 0.009892
  Cluster 4 (Manhattan): JSD = 0.002394
  Cluster 5 (Brooklyn): JSD = 0.002177
  Saved: ext7_temporal_jsd_by_cluster.html
[7.2] Hourly compositional breakdown...
  Saved: ext7_hourly_composition_by_cluster.html

[8] GRADUATED PREDICTION SCORECARD


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/1721431353.py:149: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/1721431353.py:149: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



  Saved: ext8_graduated_scorecard.html

[9] BOOTSTRAP GINI CONFIDENCE INTERVALS
[9.1] Bootstrapping 2018 Gini...
[9.2] Bootstrapping 2025 Gini...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/1721431353.py:761: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/1721431353.py:761: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.




  2018 Gini: 0.5109 [95% CI: 0.4754, 0.5414]
  2025 Gini: 0.4401 [95% CI: 0.4016, 0.4731]
  CIs overlap: False
  Saved: ext9_bootstrap_gini.html

[10] PICKUP/DROPOFF MISMATCH STATISTICS
  KS test: stat=0.2085, p=2.4268e-05
  Levene test: stat=0.0033, p=9.5430e-01
  2018 ratio: mean=0.498, std=0.054
  2025 ratio: mean=0.507, std=0.055
  Saved: ext10_mismatch_ratios.html
  Saved: ext10_mismatch_change_map.html

PART 2 COMPLETE

Outputs saved to: /Users/leoss/Desktop/Portfolio/Website-/projects/uber/outputs/
  Extension charts:
    [OK] ext6_zone_localization_distribution.html
    [OK] ext6_localization_change_map.html
    [OK] ext7_temporal_jsd_by_cluster.html
    [OK] ext7_hourly_composition_by_cluster.html
    [OK] ext8_graduated_scorecard.html
    [OK] ext9_bootstrap_gini.html
    [OK] ext10_mismatch_ratios.html
    [OK] ext10_mismatch_change_map.html

--- Methods Summary ---
  6. Zone localization: % of DO trips within same or K=5 nearest zones (Euclidean on centroids)
  7. Composit

In [ ]:
"""
Screenshots all chart HTMLs referenced in the Uber page and builds
a single self-contained HTML with embedded PNG images.

Install requirement (one-time):
    pip install playwright
    playwright install chromium

Usage:
    python screenshot_charts.py

Output:
    review_charts.html  (self-contained, small enough to upload)
"""

import os
import re
import base64
import asyncio
from playwright.async_api import async_playwright

BASE_DIR = '/Users/leoss/Desktop/Portfolio/Website-/'
PAGE_PATH = BASE_DIR + 'projects/uber/page.html'
PROJECT_DIR = BASE_DIR + 'projects/uber/'
OUTPUT_PATH = os.path.join(PROJECT_DIR, 'review_charts.html')

VIEWPORT_WIDTH = 1000
VIEWPORT_HEIGHT = 500


async def main():
    # Extract iframe sources from page
    with open(PAGE_PATH, 'r') as f:
        page_html = f.read()

    iframes = re.findall(r'<iframe\s+src="([^"]+)"', page_html)
    print(f"Found {len(iframes)} charts to screenshot")

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(viewport={'width': VIEWPORT_WIDTH, 'height': VIEWPORT_HEIGHT})

        screenshots = []
        for i, src in enumerate(iframes):
            filepath = os.path.join(PROJECT_DIR, src)
            if not os.path.exists(filepath):
                print(f"  [{i+1}/{len(iframes)}] SKIP: {src}")
                screenshots.append((src, None))
                continue

            file_url = 'file://' + os.path.abspath(filepath)
            print(f"  [{i+1}/{len(iframes)}] {src}...", end=' ')

            await page.goto(file_url, wait_until='networkidle')
            await page.wait_for_timeout(1500)  # let Plotly render

            png_bytes = await page.screenshot(full_page=True, type='png')
            b64 = base64.b64encode(png_bytes).decode('ascii')
            screenshots.append((src, b64))
            print(f"OK ({len(png_bytes) // 1024} KB)")

        await browser.close()

    # Build review HTML
    html = """<!DOCTYPE html>
<html><head><meta charset="UTF-8">
<title>Chart Review</title>
<style>
body { font-family: 'IBM Plex Sans', sans-serif; max-width: 1050px; margin: 0 auto; padding: 1rem; background: #fafafa; }
h1 { font-size: 1.3rem; }
.chart { margin-bottom: 1.5rem; background: white; border: 1px solid #ddd; border-radius: 6px; overflow: hidden; }
.chart-label { padding: 0.4rem 0.8rem; background: #f0f0f0; font-size: 0.82rem; font-weight: 600; color: #333; border-bottom: 1px solid #ddd; }
.chart img { width: 100%; display: block; }
.missing { padding: 1.5rem; color: #999; text-align: center; }
</style></head><body>
<h1>Chart Review (all charts, numbered)</h1>
"""

    for i, (src, b64) in enumerate(screenshots):
        html += f'<div class="chart">\n<div class="chart-label">#{i+1} &mdash; {src}</div>\n'
        if b64:
            html += f'<img src="data:image/png;base64,{b64}" alt="{src}">\n'
        else:
            html += f'<div class="missing">Not found: {src}</div>\n'
        html += '</div>\n'

    html += '</body></html>'

    with open(OUTPUT_PATH, 'w') as f:
        f.write(html)

    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"\nSaved: {OUTPUT_PATH}")
    print(f"Size: {size_mb:.1f} MB")
    print("Upload this file to Claude for review.")


import nest_asyncio
nest_asyncio.apply()
asyncio.run(main())

Found 29 charts to screenshot
  [1/29] outputs/ext1_market_share_stacked.html... OK (21 KB)
  [2/29] outputs/1_uber_2018_clusters.html... OK (462 KB)
  [3/29] outputs/2_uber_2025_clusters.html... OK (462 KB)
  [4/29] outputs/04_pickup_density_change.html... OK (424 KB)
  [5/29] outputs/ext3_dropoff_density_change.html... OK (430 KB)
  [6/29] outputs/11_lorenz_curve.html... OK (48 KB)
  [7/29] outputs/ext9_bootstrap_gini.html... OK (21 KB)
  [8/29] outputs/7_borough_analysis.html... OK (18 KB)
  [9/29] outputs/ext3_od_sankey_2018.html... OK (149 KB)
  [10/29] outputs/ext3_od_sankey_2025.html... OK (154 KB)
  [11/29] outputs/ext3_od_flow_heatmaps.html... OK (44 KB)
  [12/29] outputs/17_lisa_map_2018.html... OK (427 KB)
  [13/29] outputs/17_lisa_map_2025.html... OK (425 KB)
  [14/29] outputs/ext10_mismatch_ratios.html... OK (24 KB)
  [15/29] outputs/ext10_mismatch_change_map.html... OK (407 KB)
  [16/29] outputs/8_cluster_shifts_PROPER.html... OK (419 KB)
  [17/29] outputs/ext6_zone_local

In [ ]:
"""
Uber NYC Chart Fixes
====================
Regenerates charts based on review feedback. Run after the original notebook
and extension scripts have produced df_2018, df_2025, etc.

This script expects to run in the same Jupyter session where the original
notebook + extensions already ran (so all variables are in memory).
If running standalone, it reloads data.

Fixes:
- Global: white hover backgrounds on all charts
- #1:  Borough → % normalized
- #11: OD heatmap → only change panel (or improved 3-panel)
- #16: Centroid shifts → proper arrows, clear 2018/2025 markers, fixed legend
- #17: Localization distribution → % on y-axis
- #19: Trip distance → annotate small difference
- #20: Distance by cluster → fix labels, fix hover
- #21: Hourly patterns → cleaner
- #22: Demand heatmaps → per-cluster change heatmaps (2x3)
- #24: JSD → better labels/explanation
- #25: Composition → match clusters between years
- #26/27: Demand profiles → better labels
- #28/29: Demand type maps → fix hover
- #15: Mismatch map → spatial smoothing
"""

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
from scipy.stats import ks_2samp, levene, entropy
from math import radians, cos, sin, asin, sqrt
import geopandas as gpd
import libpysal
from esda.moran import Moran, Moran_Local
import json
import gc
import os

# ==========================================================================
# CONFIG
# ==========================================================================
BASE_DIR = '/Users/leoss/Desktop/Portfolio/Website-/'
DATA_DIR = BASE_DIR + 'projects/uber/data/'
OUTPUT_DIR = BASE_DIR + 'projects/uber/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_2018 = DATA_DIR + 'fhv_tripdata_2018-01.parquet'
PATH_2025 = DATA_DIR + 'fhvhv_tripdata_2025-01.parquet'
PATH_CENTROIDS = DATA_DIR + 'zone_centroids.csv'
PATH_SHAPEFILE = DATA_DIR + 'taxi_zones/taxi_zones.shp'

SAMPLE_SIZE = 20_000_000
N_CLUSTERS = 6

UBER_2018_BASES = ['B02512', 'B02598', 'B02617', 'B02682', 'B02764', 'B02765', 'B02835', 'B02836']
UBER_2025_LICENSE = 'HV0003'

STYLE = {
    'font_family': 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif',
    'tick_size': 11,
    'axis_title_size': 13,
    'legend_size': 11,
    'template': 'plotly_white',
    'plot_bg': 'rgba(0,0,0,0)',
    'paper_bg': 'white',
    'chart_height': 550,
    'margin_default': dict(l=60, r=40, t=20, b=50),
    'margin_map': dict(l=0, r=0, t=20, b=0),
    'grid_color': '#e5e7eb',
    'grid_width': 0.5,
    'year_2018': '#ff6b6b',
    'year_2025': '#4ecdc4',
    'cluster_colors': ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4'],
    'map_style': 'carto-positron-nolabels',
    'map_center': {'lat': 40.7128, 'lon': -73.9352},
    'map_zoom': 9.5,
}


def base_layout(height=None, width=None, **kwargs):
    layout = dict(
        title='',
        font=dict(family=STYLE['font_family']),
        template=STYLE['template'],
        plot_bgcolor=STYLE['plot_bg'],
        paper_bgcolor=STYLE['paper_bg'],
        height=height or STYLE['chart_height'],
        margin=STYLE['margin_default'],
        # FIXED: white hover background everywhere
        hoverlabel=dict(
            font_size=13,
            font_color='#1a2744',
            bgcolor='white',
            bordercolor='#ccc',
        ),
    )
    if width:
        layout['width'] = width
    layout.update(kwargs)
    return layout


def styled_axis(**kwargs):
    return dict(
        tickfont=dict(size=STYLE['tick_size']),
        title_font=dict(size=STYLE['axis_title_size']),
        gridcolor=STYLE['grid_color'],
        gridwidth=STYLE['grid_width'],
        **kwargs,
    )


def save_html(fig, filename):
    fig.write_html(
        OUTPUT_DIR + filename,
        include_plotlyjs='cdn',
        config={'displayModeBar': False},
    )
    print(f"  Saved: {filename}")


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * 6371 * asin(sqrt(a))


def name_cluster_by_location(center_lat, center_lon, zone_centroids):
    dists = np.sqrt((zone_centroids['latitude']-center_lat)**2 + (zone_centroids['longitude']-center_lon)**2)
    nearest = zone_centroids.iloc[dists.idxmin()]
    return f"{nearest['borough']}: {nearest['zone_name']}"


def match_clusters_by_proximity(ca, cb):
    dists = cdist(ca, cb, metric='cityblock')
    matches = {}; used = set()
    pairs = sorted([(i,j,dists[i,j]) for i in range(len(ca)) for j in range(len(cb))], key=lambda x: x[2])
    for i,j,d in pairs:
        if i not in matches and j not in used:
            matches[i] = j; used.add(j)
    return matches


def merge_zone_info(df, zone_col, centroids, prefix):
    merged = df.merge(centroids[['zone_id','zone_name','borough','latitude','longitude']], left_on=zone_col, right_on='zone_id', how='left')
    return merged.rename(columns={'zone_id':f'{prefix}_zone_id','zone_name':f'{prefix}_zone_name','borough':f'{prefix}_borough','latitude':f'{prefix}_lat','longitude':f'{prefix}_lon'})


def short_label(name):
    return name.split(':')[0].strip()


def make_short_labels(cluster_names, n_clusters):
    labels = {}; seen = {}
    for i in range(n_clusters):
        base = short_label(cluster_names[i])
        if base in seen:
            seen[base] += 1; labels[i] = f"{base} ({seen[base]})"
        else:
            seen[base] = 1; labels[i] = base
    return labels


# ==========================================================================
# LOAD DATA
# ==========================================================================
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

zone_centroids = pd.read_csv(PATH_CENTROIDS)
gdf_raw = gpd.read_file(PATH_SHAPEFILE).to_crs(epsg=4326)
taxi_zones_geo_4326 = json.loads(gdf_raw.to_json())
for f in taxi_zones_geo_4326['features']:
    f['properties']['LocationID'] = str(int(f['properties']['LocationID']))
all_zone_ids = [f['properties']['LocationID'] for f in taxi_zones_geo_4326['features']]

# 2018
print("[Loading 2018]...")
table = pq.read_table(PATH_2018, columns=['pickup_datetime','PUlocationID','DOlocationID','dispatching_base_num'])
df_2018 = table.to_pandas()
total_2018 = len(df_2018)
df_2018 = df_2018[df_2018['dispatching_base_num'].isin(UBER_2018_BASES)].copy()
uber_n_2018 = len(df_2018)
if len(df_2018) > SAMPLE_SIZE: df_2018 = df_2018.sample(n=SAMPLE_SIZE, random_state=42)
df_2018['pickup_datetime'] = pd.to_datetime(df_2018['pickup_datetime'])
df_2018['hour'] = df_2018['pickup_datetime'].dt.hour
df_2018['day_of_week'] = df_2018['pickup_datetime'].dt.dayofweek
df_2018['day_name'] = df_2018['pickup_datetime'].dt.day_name()
df_2018 = df_2018.dropna(subset=['PUlocationID'])
df_2018['PUlocationID'] = df_2018['PUlocationID'].astype(int)
df_2018 = merge_zone_info(df_2018, 'PUlocationID', zone_centroids, 'PU')
df_2018 = df_2018.dropna(subset=['PU_lat','PU_lon'])
df_2018['DOlocationID'] = pd.to_numeric(df_2018['DOlocationID'], errors='coerce')
df_2018.loc[df_2018['DOlocationID'].notna(), 'DOlocationID'] = df_2018.loc[df_2018['DOlocationID'].notna(), 'DOlocationID'].astype(int)
df_2018 = merge_zone_info(df_2018, 'DOlocationID', zone_centroids, 'DO')
kmeans_2018 = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_2018['PU_cluster'] = kmeans_2018.fit_predict(df_2018[['PU_lat','PU_lon']].values)
cnames_2018 = {i: name_cluster_by_location(*kmeans_2018.cluster_centers_[i], zone_centroids) for i in range(N_CLUSTERS)}
df_2018['PU_cluster_name'] = df_2018['PU_cluster'].map(cnames_2018)
do_mask = df_2018['DO_lat'].notna() & df_2018['DO_lon'].notna()
df_2018.loc[do_mask, 'DO_cluster'] = kmeans_2018.predict(df_2018.loc[do_mask, ['DO_lat','DO_lon']].values)
short_labels_2018 = make_short_labels(cnames_2018, N_CLUSTERS)
del table; gc.collect()
print(f"  2018: {len(df_2018):,} trips")

# 2025
print("[Loading 2025]...")
table = pq.read_table(PATH_2025, columns=['pickup_datetime','PULocationID','DOLocationID','hvfhs_license_num'])
df_2025 = table.to_pandas()
total_2025 = len(df_2025)
df_2025 = df_2025[df_2025['hvfhs_license_num'] == UBER_2025_LICENSE].copy()
uber_n_2025 = len(df_2025)
if len(df_2025) > SAMPLE_SIZE: df_2025 = df_2025.sample(n=SAMPLE_SIZE, random_state=42)
df_2025['pickup_datetime'] = pd.to_datetime(df_2025['pickup_datetime'])
df_2025['hour'] = df_2025['pickup_datetime'].dt.hour
df_2025['day_of_week'] = df_2025['pickup_datetime'].dt.dayofweek
df_2025['day_name'] = df_2025['pickup_datetime'].dt.day_name()
df_2025 = df_2025.dropna(subset=['PULocationID'])
df_2025['PULocationID'] = df_2025['PULocationID'].astype(int)
df_2025 = merge_zone_info(df_2025, 'PULocationID', zone_centroids, 'PU')
df_2025 = df_2025.dropna(subset=['PU_lat','PU_lon'])
df_2025['DOLocationID'] = pd.to_numeric(df_2025['DOLocationID'], errors='coerce')
df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'] = df_2025.loc[df_2025['DOLocationID'].notna(), 'DOLocationID'].astype(int)
df_2025 = merge_zone_info(df_2025, 'DOLocationID', zone_centroids, 'DO')
kmeans_2025 = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_2025['PU_cluster'] = kmeans_2025.fit_predict(df_2025[['PU_lat','PU_lon']].values)
cnames_2025 = {i: name_cluster_by_location(*kmeans_2025.cluster_centers_[i], zone_centroids) for i in range(N_CLUSTERS)}
df_2025['PU_cluster_name'] = df_2025['PU_cluster'].map(cnames_2025)
do_mask = df_2025['DO_lat'].notna() & df_2025['DO_lon'].notna()
df_2025.loc[do_mask, 'DO_cluster'] = kmeans_2025.predict(df_2025.loc[do_mask, ['DO_lat','DO_lon']].values)
short_labels_2025 = make_short_labels(cnames_2025, N_CLUSTERS)
del table; gc.collect()
print(f"  2025: {len(df_2025):,} trips")

centers_2018 = kmeans_2018.cluster_centers_
centers_2025 = kmeans_2025.cluster_centers_
cluster_matches = match_clusters_by_proximity(centers_2018, centers_2025)


# ══════════════════════════════════════════════════════════════════════════
# FIX: BOROUGH ANALYSIS (% normalized)
# ══════════════════════════════════════════════════════════════════════════
print("\n[FIX] Borough analysis (% normalized)...")

borough_2018 = df_2018.groupby('PU_borough').size().reset_index(name='trips')
borough_2025 = df_2025.groupby('PU_borough').size().reset_index(name='trips')
borough_2018['pct'] = 100 * borough_2018['trips'] / borough_2018['trips'].sum()
borough_2025['pct'] = 100 * borough_2025['trips'] / borough_2025['trips'].sum()
borough_df = borough_2018[['PU_borough','pct']].rename(columns={'pct':'pct_2018'}).merge(
    borough_2025[['PU_borough','pct']].rename(columns={'pct':'pct_2025'}), on='PU_borough'
)

fig = go.Figure()
fig.add_trace(go.Bar(x=borough_df['PU_borough'], y=borough_df['pct_2018'], name='2018',
    marker_color=STYLE['year_2018'],
    text=[f"{v:.1f}%" for v in borough_df['pct_2018']], textposition='outside',
    hovertemplate='%{x}<br>2018: %{y:.1f}%<extra></extra>'))
fig.add_trace(go.Bar(x=borough_df['PU_borough'], y=borough_df['pct_2025'], name='2025',
    marker_color=STYLE['year_2025'],
    text=[f"{v:.1f}%" for v in borough_df['pct_2025']], textposition='outside',
    hovertemplate='%{x}<br>2025: %{y:.1f}%<extra></extra>'))
fig.update_layout(**base_layout(height=500, width=900),
    xaxis=styled_axis(title_text='Borough'),
    yaxis=styled_axis(title_text='Share of Pickups (%)', range=[0, 80]),
    barmode='group',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_borough_pct.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: CENTROID SHIFTS (proper arrows, clear markers, fixed legend)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Centroid shifts...")

shift_data = []
for i18, i25 in cluster_matches.items():
    lat1, lon1 = centers_2018[i18]
    lat2, lon2 = centers_2025[i25]
    shift_data.append({
        'idx_18': i18, 'idx_25': i25,
        'name_18': cnames_2018[i18], 'name_25': cnames_2025[i25],
        'dist_km': haversine_km(lat1, lon1, lat2, lon2)
    })

fig_shift = go.Figure()

# Background zones
fig_shift.add_trace(go.Choroplethmap(
    geojson=taxi_zones_geo_4326, locations=all_zone_ids,
    featureidkey='properties.LocationID',
    z=[1]*len(all_zone_ids), colorscale=[[0,'#e8e8e8'],[1,'#e8e8e8']],
    marker_opacity=0.35, marker_line_width=0.3, marker_line_color='#ccc',
    showscale=False, hoverinfo='skip'))

# Shift lines with midpoint arrows
for sa in shift_data:
    lat1, lon1 = centers_2018[sa['idx_18']]
    lat2, lon2 = centers_2025[sa['idx_25']]
    # Draw line
    fig_shift.add_trace(go.Scattermap(
        lat=[lat1, lat2], lon=[lon1, lon2],
        mode='lines', line=dict(width=3, color='#333'),
        showlegend=False,
        hovertemplate=(f"<b>{short_label(sa['name_18'])}</b><br>"
                       f"Shift: {sa['dist_km']:.2f} km<extra></extra>")))
    # Midpoint arrow marker
    mid_lat = (lat1 + lat2) / 2
    mid_lon = (lon1 + lon2) / 2
    fig_shift.add_trace(go.Scattermap(
        lat=[mid_lat + (lat2-lat1)*0.15], lon=[mid_lon + (lon2-lon1)*0.15],
        mode='markers', marker=dict(size=8, color='#333', symbol='triangle'),
        showlegend=False, hoverinfo='skip'))

# 2018 centroids (filled circles)
for i in range(N_CLUSTERS):
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2018[i,0]], lon=[centers_2018[i,1]],
        mode='markers+text',
        marker=dict(size=14, color=STYLE['cluster_colors'][i], opacity=0.95,
                    symbol='circle'),
        text=short_labels_2018[i],
        textfont=dict(size=9, color='#333', family=STYLE['font_family']),
        textposition='top center',
        name=f'2018: {short_labels_2018[i]}',
        legendgroup='2018', legendgrouptitle_text='2018',
        hovertemplate=f'<b>2018: {short_labels_2018[i]}</b><extra></extra>'))

# 2025 centroids (diamond markers)
for i in range(N_CLUSTERS):
    matched_18 = [k for k, v in cluster_matches.items() if v == i]
    color_idx = matched_18[0] if matched_18 else i
    fig_shift.add_trace(go.Scattermap(
        lat=[centers_2025[i,0]], lon=[centers_2025[i,1]],
        mode='markers',
        marker=dict(size=14, color=STYLE['cluster_colors'][color_idx],
                    opacity=0.95, symbol='square'),
        name=f'2025: {short_labels_2025[i]}',
        legendgroup='2025', legendgrouptitle_text='2025',
        hovertemplate=f'<b>2025: {short_labels_2025[i]}</b><extra></extra>'))

fig_shift.update_layout(
    **base_layout(height=700, width=1100, margin=STYLE['margin_map']),
    map_style=STYLE['map_style'], map_zoom=9.8,
    map_center=dict(lat=40.72, lon=-73.94),
    legend=dict(
        yanchor='top', y=0.99, xanchor='left', x=0.01,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#dde1e7', borderwidth=1,
        font=dict(size=11, family=STYLE['font_family']),
        itemsizing='constant'))
save_html(fig_shift, 'fix_cluster_shifts.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: HOURLY PATTERNS (cleaner)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Hourly patterns...")

h18 = df_2018.groupby('hour').size()
h25 = df_2025.groupby('hour').size()
h18_pct = 100 * h18 / h18.sum()
h25_pct = 100 * h25 / h25.sum()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=h18_pct.index, y=h18_pct.values, name='2018', mode='lines',
    line=dict(color=STYLE['year_2018'], width=2.5, shape='spline'),
    fill='tozeroy', fillcolor='rgba(255,107,107,0.1)',
    hovertemplate='Hour %{x}:00<br>%{y:.1f}% of trips<extra>2018</extra>'))
fig.add_trace(go.Scatter(
    x=h25_pct.index, y=h25_pct.values, name='2025', mode='lines',
    line=dict(color=STYLE['year_2025'], width=2.5, shape='spline'),
    fill='tozeroy', fillcolor='rgba(78,205,196,0.1)',
    hovertemplate='Hour %{x}:00<br>%{y:.1f}% of trips<extra>2025</extra>'))
fig.update_layout(**base_layout(height=450, width=900),
    xaxis=styled_axis(title_text='Hour of Day', dtick=2),
    yaxis=styled_axis(title_text='Share of Daily Trips (%)'),
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_hourly_patterns.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: DEMAND HEATMAP (per-cluster change, 2x3 grid)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Per-cluster temporal change heatmaps...")

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig_heat = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f"{short_labels_2018[i]}" for i in range(N_CLUSTERS)],
    shared_xaxes=True, shared_yaxes=True,
    vertical_spacing=0.08, horizontal_spacing=0.04)

for i in range(N_CLUSTERS):
    row = i // 3 + 1
    col = i % 3 + 1

    # 2018 cluster
    c18 = df_2018[df_2018['PU_cluster'] == i]
    piv18 = c18.groupby(['day_name','hour']).size().reset_index(name='trips')
    piv18 = piv18.pivot(index='day_name', columns='hour', values='trips').reindex(day_order).fillna(0)
    piv18_pct = 100 * piv18 / piv18.sum().sum()

    # Matched 2025 cluster
    i25 = cluster_matches[i]
    c25 = df_2025[df_2025['PU_cluster'] == i25]
    piv25 = c25.groupby(['day_name','hour']).size().reset_index(name='trips')
    piv25 = piv25.pivot(index='day_name', columns='hour', values='trips').reindex(day_order).fillna(0)
    piv25_pct = 100 * piv25 / piv25.sum().sum()

    diff = piv25_pct.values - piv18_pct.values

    fig_heat.add_trace(go.Heatmap(
        z=diff, x=list(range(24)), y=day_order,
        colorscale='RdBu', zmid=0,
        showscale=(i == N_CLUSTERS - 1),
        colorbar=dict(
            title=dict(text='Change (pp)'), ticksuffix=' pp', x=1.02,
            tickfont=dict(family=STYLE['font_family'])) if i == N_CLUSTERS - 1 else None,
        hovertemplate='<b>%{y}, %{x}:00</b><br>Change: %{z:.3f} pp<extra></extra>',
    ), row=row, col=col)

fig_heat.update_layout(**base_layout(height=500, width=1200))
fig_heat.update_xaxes(title_text='Hour', row=2)
fig_heat.update_yaxes(tickfont=dict(size=9))
save_html(fig_heat, 'fix_cluster_heatmap_changes.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: LOCALIZATION DISTRIBUTION (% y-axis)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Localization distribution (%)...")

K_NEAREST = 5
zc = zone_centroids.drop_duplicates(subset='zone_id').set_index('zone_id')
zone_ids_all = zc.index.values
zone_coords = zc[['latitude','longitude']].values
zone_dist_matrix = cdist(zone_coords, zone_coords, metric='euclidean')
nearest_zones = {}
for i, zid in enumerate(zone_ids_all):
    sorted_idx = np.argsort(zone_dist_matrix[i])
    nearest_zones[zid] = set([zone_ids_all[j] for j in sorted_idx[1:K_NEAREST+1]]) | {zid}

def compute_zone_localization(df, nz, pu='PU_zone_id', do='DO_zone_id'):
    valid = df.dropna(subset=[pu, do]).copy()
    valid[pu] = valid[pu].astype(int); valid[do] = valid[do].astype(int)
    od = valid.groupby([pu, do]).size().reset_index(name='trips')
    od['is_local'] = od.apply(lambda r: int(r[do]) in nz.get(int(r[pu]), set()), axis=1)
    total = od.groupby(pu)['trips'].sum().rename('total')
    local = od[od['is_local']].groupby(pu)['trips'].sum().rename('local')
    zl = pd.concat([total, local], axis=1).fillna(0)
    zl['local_share'] = 100 * zl['local'] / zl['total']
    return zl

local_2018 = compute_zone_localization(df_2018, nearest_zones)
local_2025 = compute_zone_localization(df_2025, nearest_zones)

fig = go.Figure()
for label, data, color in [('2018', local_2018, STYLE['year_2018']), ('2025', local_2025, STYLE['year_2025'])]:
    fig.add_trace(go.Histogram(x=data['local_share'], histnorm='percent', nbinsx=40,
        name=label, marker_color=color, opacity=0.6,
        hovertemplate='%{x:.0f}%: %{y:.1f}% of zones<extra>'+label+'</extra>'))

fig.add_vline(x=local_2018['local_share'].mean(), line_dash='dash', line_color=STYLE['year_2018'], line_width=2,
    annotation_text=f"2018 mean: {local_2018['local_share'].mean():.1f}%", annotation_position='top left')
fig.add_vline(x=local_2025['local_share'].mean(), line_dash='dash', line_color=STYLE['year_2025'], line_width=2,
    annotation_text=f"2025 mean: {local_2025['local_share'].mean():.1f}%", annotation_position='top right')
fig.update_layout(**base_layout(height=450, width=900),
    xaxis=styled_axis(title_text=f'% of Trips to Same or {K_NEAREST}-Nearest Zones'),
    yaxis=styled_axis(title_text='% of Zones'),
    barmode='overlay',
    legend=dict(x=0.75, y=0.95, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_localization_pct.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: TRIP DISTANCE (annotate small difference)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Trip distance distribution...")

def haversine_vec(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (np.radians(x) for x in [lat1,lon1,lat2,lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

for label, df in [('2018', df_2018), ('2025', df_2025)]:
    v = df.dropna(subset=['PU_lat','PU_lon','DO_lat','DO_lon'])
    df.loc[v.index, 'trip_dist_km'] = haversine_vec(v['PU_lat'].values, v['PU_lon'].values, v['DO_lat'].values, v['DO_lon'].values)

d18 = df_2018[(df_2018['trip_dist_km'] > 0) & (df_2018['trip_dist_km'] < 100)]['trip_dist_km']
d25 = df_2025[(df_2025['trip_dist_km'] > 0) & (df_2025['trip_dist_km'] < 100)]['trip_dist_km']

fig = go.Figure()
fig.add_trace(go.Histogram(x=d18, histnorm='probability density', nbinsx=80, name='2018',
    marker_color=STYLE['year_2018'], opacity=0.6,
    hovertemplate='%{x:.1f} km<br>Density: %{y:.4f}<extra>2018</extra>'))
fig.add_trace(go.Histogram(x=d25, histnorm='probability density', nbinsx=80, name='2025',
    marker_color=STYLE['year_2025'], opacity=0.6,
    hovertemplate='%{x:.1f} km<br>Density: %{y:.4f}<extra>2025</extra>'))
fig.update_layout(**base_layout(height=450, width=900),
    xaxis=styled_axis(title_text='Trip Distance (km)', range=[0, 30]),
    yaxis=styled_axis(title_text='Density'),
    barmode='overlay',
    legend=dict(x=0.75, y=0.95, font=dict(family=STYLE['font_family'])),
    annotations=[dict(
        x=0.98, y=0.95, xref='paper', yref='paper', xanchor='right',
        text=(f"Median: {d18.median():.2f} km (2018) vs {d25.median():.2f} km (2025)<br>"
              f"Mean: {d18.mean():.2f} km vs {d25.mean():.2f} km<br>"
              f"Distributions are nearly identical"),
        showarrow=False, font=dict(size=11, family=STYLE['font_family']),
        bgcolor='rgba(255,255,255,0.9)', bordercolor='#ddd', borderwidth=1)])
save_html(fig, 'fix_trip_distance.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: DISTANCE BY CLUSTER (fix labels, fix hover)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Distance by cluster...")

dist_data = []
for i in range(N_CLUSTERS):
    d18 = df_2018[(df_2018['PU_cluster']==i) & (df_2018['trip_dist_km']>0) & (df_2018['trip_dist_km']<100)]['trip_dist_km']
    i25 = cluster_matches[i]
    d25_v = df_2025[(df_2025['PU_cluster']==i25) & (df_2025['trip_dist_km']>0) & (df_2025['trip_dist_km']<100)]['trip_dist_km']
    dist_data.append({
        'cluster': short_labels_2018[i],
        'median_2018': d18.median() if len(d18) > 0 else 0,
        'median_2025': d25_v.median() if len(d25_v) > 0 else 0,
    })

ddf = pd.DataFrame(dist_data)
fig = go.Figure()
fig.add_trace(go.Bar(x=ddf['cluster'], y=ddf['median_2018'], name='2018 (median)',
    marker_color=STYLE['year_2018'],
    text=[f"{v:.1f}" for v in ddf['median_2018']], textposition='outside',
    hovertemplate='%{x}<br>2018 median: %{y:.2f} km<extra></extra>'))
fig.add_trace(go.Bar(x=ddf['cluster'], y=ddf['median_2025'], name='2025 (median)',
    marker_color=STYLE['year_2025'],
    text=[f"{v:.1f}" for v in ddf['median_2025']], textposition='outside',
    hovertemplate='%{x}<br>2025 median: %{y:.2f} km<extra></extra>'))
fig.update_layout(**base_layout(height=450, width=900),
    xaxis=styled_axis(title_text='Pickup Cluster'),
    yaxis=styled_axis(title_text='Median Trip Distance (km)'),
    barmode='group',
    legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])))
save_html(fig, 'fix_distance_by_cluster.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: JSD BY CLUSTER (better labels)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] JSD by cluster...")

def hourly_profile(df, col=None, val=None):
    if col: df = df[df[col]==val]
    c = df.groupby('hour').size().reindex(range(24), fill_value=0)
    t = c.sum()
    return (c / t).values if t > 0 else np.ones(24)/24

def jsd(p, q):
    eps = 1e-12
    p = np.array(p)+eps; q = np.array(q)+eps
    p /= p.sum(); q /= q.sum()
    m = 0.5*(p+q)
    return 0.5*(entropy(p,m,base=2) + entropy(q,m,base=2))

agg_jsd = jsd(hourly_profile(df_2018), hourly_profile(df_2025))
cluster_jsds = []
for i in range(N_CLUSTERS):
    i25 = cluster_matches[i]
    d = jsd(hourly_profile(df_2018,'PU_cluster',i), hourly_profile(df_2025,'PU_cluster',i25))
    cluster_jsds.append({'cluster': short_labels_2018[i], 'jsd': d, 'idx': i})

cjdf = pd.DataFrame(cluster_jsds)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=cjdf['cluster'], y=cjdf['jsd'],
    marker_color=[STYLE['cluster_colors'][r['idx'] % len(STYLE['cluster_colors'])] for _, r in cjdf.iterrows()],
    hovertemplate='%{x}<br>JSD: %{y:.5f}<br><i>Higher = more temporal change</i><extra></extra>'))
fig.add_hline(y=agg_jsd, line_dash='dash', line_color='#333', line_width=2,
    annotation_text=f'Citywide average: {agg_jsd:.4f}', annotation_position='top right')
fig.update_layout(**base_layout(height=450, width=850),
    xaxis=styled_axis(title_text='Geographic Cluster'),
    yaxis=styled_axis(title_text='Temporal Shift (JSD)',
        tickformat='.4f'),
    showlegend=False,
    annotations=[dict(
        x=0.5, y=-0.18, xref='paper', yref='paper',
        text="JSD measures how much the hourly demand profile changed between 2018 and 2025. Values near 0 = no change.",
        showarrow=False, font=dict(size=10, color='#777', family=STYLE['font_family']))])
save_html(fig, 'fix_jsd_by_cluster.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: COMPOSITION BY HOUR (matched clusters)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Hourly composition (matched clusters)...")

fig_comp = make_subplots(rows=1, cols=2,
    subplot_titles=('2018', '2025'), horizontal_spacing=0.08)

# Use consistent ordering: sort 2018 clusters by size
cluster_order = sorted(range(N_CLUSTERS), key=lambda i: -(df_2018['PU_cluster']==i).sum())

for col_idx, (df_year, year_label) in enumerate([(df_2018, '2018'), (df_2025, '2025')], 1):
    hourly_by_cluster = df_year.groupby(['hour','PU_cluster']).size().reset_index(name='trips')
    hourly_total = df_year.groupby('hour').size().reset_index(name='total')
    hourly_by_cluster = hourly_by_cluster.merge(hourly_total, on='hour')
    hourly_by_cluster['share'] = 100 * hourly_by_cluster['trips'] / hourly_by_cluster['total']

    for c in cluster_order:
        if year_label == '2018':
            cluster_idx = c
            label = short_labels_2018[c]
            color = STYLE['cluster_colors'][c % len(STYLE['cluster_colors'])]
        else:
            cluster_idx = cluster_matches[c]
            label = short_labels_2018[c]  # Use 2018 label for consistency
            color = STYLE['cluster_colors'][c % len(STYLE['cluster_colors'])]

        subset = hourly_by_cluster[hourly_by_cluster['PU_cluster'] == cluster_idx]
        fig_comp.add_trace(go.Bar(
            x=subset['hour'], y=subset['share'],
            name=label, marker_color=color,
            legendgroup=str(c), showlegend=(col_idx == 1),
            hovertemplate=f'{label}<br>Hour %{{x}}:00: %{{y:.1f}}%<extra></extra>',
        ), row=1, col=col_idx)

fig_comp.update_layout(**base_layout(height=450, width=1100),
    barmode='stack',
    legend=dict(x=0.85, y=0.98, font=dict(family=STYLE['font_family'])))
fig_comp.update_xaxes(title_text='Hour of Day')
fig_comp.update_yaxes(title_text='Cluster Share (%)', col=1)
save_html(fig_comp, 'fix_hourly_composition.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: DEMAND PROFILES (better labels)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] Demand profiles...")

demand_colors = ['#264653', '#2a9d8f', '#e9c46a', '#e76f51']
N_DEMAND = 4

for year_label, df_year in [('2018', df_2018), ('2025', df_2025)]:
    hourly = df_year.groupby(['PU_zone_id','hour']).size().reset_index(name='trips')
    pivot = hourly.pivot(index='PU_zone_id', columns='hour', values='trips').fillna(0)
    row_sums = pivot.sum(axis=1)
    pivot_norm = pivot.div(row_sums, axis=0).fillna(0)

    zone_totals = df_year.groupby('PU_zone_id').size()
    valid = zone_totals[zone_totals >= 50].index
    pivot_norm = pivot_norm.loc[pivot_norm.index.isin(valid)]

    X = StandardScaler().fit_transform(pivot_norm.values)
    km = KMeans(n_clusters=N_DEMAND, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    pivot_norm['dc'] = labels

    dc_names = {}
    for c in range(N_DEMAND):
        mask = pivot_norm['dc'] == c
        avg = pivot_norm.loc[mask].drop(columns='dc').mean()
        peak = avg.idxmax()
        n = mask.sum()
        if peak <= 9: pat = 'Morning-peak'
        elif peak <= 14: pat = 'Midday'
        elif peak <= 19: pat = 'Evening-peak'
        else: pat = 'Night-peak'
        dc_names[c] = f"{pat} ({n} zones, peak ~{peak}:00)"

    fig = go.Figure()
    for c in range(N_DEMAND):
        mask = pivot_norm['dc'] == c
        avg = pivot_norm.loc[mask].drop(columns=['dc']).mean()
        fig.add_trace(go.Scatter(
            x=list(range(24)), y=avg.values,
            mode='lines+markers', name=dc_names[c],
            line=dict(color=demand_colors[c % len(demand_colors)], width=2.5),
            marker=dict(size=5),
            hovertemplate='Hour %{x}:00<br>Share: %{y:.3f}<extra>'+dc_names[c]+'</extra>'))

    fig.update_layout(**base_layout(height=450, width=900),
        xaxis=styled_axis(title_text='Hour of Day', dtick=2),
        yaxis=styled_axis(title_text='Within-Zone Trip Share'),
        legend=dict(x=0.02, y=0.98, font=dict(family=STYLE['font_family'])),
        annotations=[dict(
            x=0.5, y=-0.18, xref='paper', yref='paper',
            text=f"Zones grouped by when their demand peaks. Each line is the average hourly profile for that group ({year_label}).",
            showarrow=False, font=dict(size=10, color='#777', family=STYLE['font_family']))])
    save_html(fig, f'fix_demand_profiles_{year_label}.html')

    # Demand type maps with visible hover
    zone_demand = pivot_norm[['dc']].reset_index()
    zone_demand.columns = ['zone_id', 'dc']
    zone_demand['dc_name'] = zone_demand['dc'].map(dc_names)
    zone_demand['zone_id'] = zone_demand['zone_id'].astype(float).astype(int).astype(str)
    zc_lu = zone_centroids.drop_duplicates(subset='zone_id').set_index('zone_id')
    zone_demand['zone_name'] = zone_demand['zone_id'].astype(int).map(zc_lu['zone_name'])
    zone_demand['borough'] = zone_demand['zone_id'].astype(int).map(zc_lu['borough'])

    zids = set(zone_demand['zone_id'].values)
    fgeo = {'type':'FeatureCollection',
            'features':[f for f in taxi_zones_geo_4326['features'] if f['properties']['LocationID'] in zids]}

    dc_color_map = {dc_names[c]: demand_colors[c % len(demand_colors)] for c in range(N_DEMAND)}

    fig_map = go.Figure()
    fig_map.add_trace(go.Choroplethmap(
        geojson=taxi_zones_geo_4326, locations=all_zone_ids,
        featureidkey='properties.LocationID',
        z=[1]*len(all_zone_ids), colorscale=[[0,'#e8e8e8'],[1,'#e8e8e8']],
        marker_opacity=0.4, marker_line_width=0.3, marker_line_color='#ccc',
        showscale=False, hoverinfo='skip'))

    for c in range(N_DEMAND):
        subset = zone_demand[zone_demand['dc'] == c]
        if len(subset) == 0: continue
        sgeo = {'type':'FeatureCollection',
                'features':[f for f in fgeo['features']
                            if f['properties']['LocationID'] in set(subset['zone_id'].values)]}
        fig_map.add_trace(go.Choroplethmap(
            geojson=sgeo, locations=subset['zone_id'].values,
            featureidkey='properties.LocationID',
            z=[1]*len(subset), colorscale=[[0,demand_colors[c]],[1,demand_colors[c]]],
            marker_opacity=0.8, marker_line_width=0.5,
            marker_line_color='rgba(255,255,255,0.7)',
            showscale=False, name=dc_names[c], showlegend=True,
            customdata=np.column_stack([
                subset['zone_name'].fillna('Unknown').values,
                subset['borough'].fillna('Unknown').values,
                subset['dc_name'].values]),
            hovertemplate='<b>%{customdata[0]}</b> (%{customdata[1]})<br>Type: %{customdata[2]}<extra></extra>'))

    fig_map.update_layout(
        **base_layout(height=650, width=1100, margin=STYLE['margin_map']),
        legend=dict(title="Demand Type", yanchor="top", y=0.99, xanchor="left", x=0.01,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#dde1e7", borderwidth=1,
            font=dict(size=10, family=STYLE['font_family'])),
        map=dict(style=STYLE['map_style'], center=STYLE['map_center'], zoom=STYLE['map_zoom']))
    save_html(fig_map, f'fix_demand_map_{year_label}.html')


# ══════════════════════════════════════════════════════════════════════════
# FIX: OD HEATMAP (improved, change-focused)
# ══════════════════════════════════════════════════════════════════════════
print("[FIX] OD flow heatmap...")

def pivot_od(df, n):
    valid = df.dropna(subset=['PU_cluster','DO_cluster']).copy()
    valid['PU_cluster'] = valid['PU_cluster'].astype(int)
    valid['DO_cluster'] = valid['DO_cluster'].astype(int)
    od = valid.groupby(['PU_cluster','DO_cluster']).size().reset_index(name='trips')
    total = od['trips'].sum()
    matrix = np.zeros((n, n))
    for _, r in od.iterrows():
        if int(r['PU_cluster']) < n and int(r['DO_cluster']) < n:
            matrix[int(r['PU_cluster']), int(r['DO_cluster'])] = 100 * r['trips'] / total
    return matrix

od18 = pivot_od(df_2018, N_CLUSTERS)
od25 = pivot_od(df_2025, N_CLUSTERS)

# Align 2025 to 2018 cluster order
od25_aligned = np.zeros_like(od25)
for i18, i25 in cluster_matches.items():
    for j18, j25 in cluster_matches.items():
        od25_aligned[i18, j18] = od25[i25, j25]

od_diff = od25_aligned - od18
cl_short = [short_labels_2018[i] for i in range(N_CLUSTERS)]

fig = go.Figure(go.Heatmap(
    z=od_diff, x=cl_short, y=cl_short,
    colorscale='RdBu', zmid=0,
    colorbar=dict(title=dict(text='Change (pp)'), ticksuffix=' pp',
                  tickfont=dict(family=STYLE['font_family'])),
    hovertemplate='PU: %{y}<br>DO: %{x}<br>Change: %{z:.2f} pp<extra></extra>',
    text=np.round(od_diff, 2), texttemplate='%{text:.1f}',
    textfont=dict(size=10)))
fig.update_layout(**base_layout(height=500, width=600),
    xaxis=styled_axis(title_text='Dropoff Cluster'),
    yaxis=styled_axis(title_text='Pickup Cluster'))
save_html(fig, 'fix_od_change.html')


# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("ALL FIXES COMPLETE")
print("=" * 70)
fixes = [
    'fix_borough_pct.html',
    'fix_cluster_shifts.html',
    'fix_hourly_patterns.html',
    'fix_cluster_heatmap_changes.html',
    'fix_localization_pct.html',
    'fix_trip_distance.html',
    'fix_distance_by_cluster.html',
    'fix_jsd_by_cluster.html',
    'fix_hourly_composition.html',
    'fix_demand_profiles_2018.html',
    'fix_demand_profiles_2025.html',
    'fix_demand_map_2018.html',
    'fix_demand_map_2025.html',
    'fix_od_change.html',
]
for f in fixes:
    status = "OK" if os.path.exists(OUTPUT_DIR + f) else "MISSING"
    print(f"  [{status}] {f}")

LOADING DATA
[Loading 2018]...


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pandas/core/arrays/datetimes.py:666: RuntimeWarning:

coroutine 'main' was never awaited



  2018: 4,519,809 trips
[Loading 2025]...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_10429/2284126892.py:232: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[211 181  76 ... 235 235 133]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.



  2025: 15,447,196 trips

[FIX] Borough analysis (% normalized)...
  Saved: fix_borough_pct.html
[FIX] Centroid shifts...
  Saved: fix_cluster_shifts.html
[FIX] Hourly patterns...
  Saved: fix_hourly_patterns.html
[FIX] Per-cluster temporal change heatmaps...
  Saved: fix_cluster_heatmap_changes.html
[FIX] Localization distribution (%)...
  Saved: fix_localization_pct.html
[FIX] Trip distance distribution...
  Saved: fix_trip_distance.html
[FIX] Distance by cluster...
  Saved: fix_distance_by_cluster.html
[FIX] JSD by cluster...
  Saved: fix_jsd_by_cluster.html
[FIX] Hourly composition (matched clusters)...
  Saved: fix_hourly_composition.html
[FIX] Demand profiles...
  Saved: fix_demand_profiles_2018.html
  Saved: fix_demand_map_2018.html
  Saved: fix_demand_profiles_2025.html
  Saved: fix_demand_map_2025.html
[FIX] OD flow heatmap...
  Saved: fix_od_change.html

ALL FIXES COMPLETE
  [OK] fix_borough_pct.html
  [OK] fix_cluster_shifts.html
  [OK] fix_hourly_patterns.html
  [OK] fix_cl